# Biological BMI Paper — LASSO Modeling for BMI with Sex-stratified Version

***by Kengo Watanabe***  

In this notebook, LASSO models for BMI (biological BMIs) are generated from the baseline Arivale datasets for biological BMI paper. In addition, sex-stratified models are generated to compare both sex model.  

Input:  
* Cleaned metabolomics: 210104_Biological-BMI-paper_RF-imputation_baseline-metDF-with-RF-imputation.tsv  
* Cleaned clinical labs: 210104_Biological-BMI-paper_RF-imputation_baseline-chemDF-with-RF-imputation.tsv  
* Cleaned proteomics: 210104_Biological-BMI-paper_RF-imputation_baseline-protDF-with-RF-imputation.tsv  
* Cleaned combined omics: 210104_Biological-BMI-paper_RF-imputation_baseline-combiDF-with-RF-imputation.tsv  

Output:  
* Figure 1a, 1b  
* Supplementary Figure 3  
* Intermediate tables for other notebook (BMI predictions, beta-coefficients)  

Original notebook:  
* 210105_Biological-BMI-paper_baseline-LASSO.ipynb  
* 210530_Biological-BMI-paper_baseline-LASSO(figure-layout).ipynb  

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
#For Arial font
#!conda install -c conda-forge -y mscorefonts
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display
import time

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV

!conda list

In [ ]:
covarL = ['log_BaseBMI', 'Sex', 'BaseAge', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'Race']

## 1. Metabolomics

### 1-1. Sex stratification

In [ ]:
#Import the cleaned baseline dataframe for LASSO
fileDir = './ExportData/'
fileName = '210104_Biological-BMI-paper_RF-imputation_baseline-metDF-with-RF-imputation.tsv'
metDF = pd.read_csv(fileDir+fileName, sep='\t', dtype={'public_client_id': str}).set_index('public_client_id')
display(metDF)

In [ ]:
#Splitting dataset into sex-dependent cohorts
metDF_F = metDF.loc[metDF['Sex']=='F']
metDF_M = metDF.loc[metDF['Sex']=='M']
metDF_B = metDF#Not copy just rename
print('(Female, Male, Both): (',
      len(metDF_F.index), ', ', len(metDF_M.index), ', ', len(metDF_B.index), ')')

### 1-2. Standarization

In [ ]:
#Female cohort
tempDF = metDF_F.drop(columns=covarL)
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()

#Z-score transformation
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
tempA = scaler.fit_transform(tempDF)#Column direction
tempDF = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)

display(tempDF)
display(tempDF.describe())

#Confirmation
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

#Update
metDF_F_x = tempDF

In [ ]:
#Male cohort
tempDF = metDF_M.drop(columns=covarL)
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()

#Z-score transformation
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
tempA = scaler.fit_transform(tempDF)#Column direction
tempDF = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)

display(tempDF)
display(tempDF.describe())

#Confirmation
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

#Update
metDF_M_x = tempDF

In [ ]:
#Both sex cohort
tempDF = metDF_B.drop(columns=covarL)
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()

#Z-score transformation
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
tempA = scaler.fit_transform(tempDF)#Column direction
tempDF = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)

display(tempDF)
display(tempDF.describe())

#Confirmation
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

#Update
metDF_B_x = tempDF

### 1-3. LASSO with cross-validation

In [ ]:
#Female model
#Split into 10 datasets
x_folds = np.array_split(metDF_F_x, 10)
tempDF = metDF_F.drop(columns=list(set(metDF_F.columns)-set(['log_BaseBMI'])))#Not Series but DF
y_folds = np.array_split(tempDF, 10)

model = LassoCV(eps=0.05, n_alphas=200, alphas=None, fit_intercept=True,
                normalize=False, precompute='auto', cv=10)
metBMI_F_bcoefs = pd.DataFrame(index=metDF_F_x.columns).astype('float64')#For beta-coefficient
metBMI_F_bcoefs.index.set_names('Variable', inplace=True)
metBMI_F_scores = []#For the coefficient of determination R^2
tempL = []#For predictions
#Perform LASSO
t_start = time.time()
for model_k in range(10):
    #Set training/testing dataset in model #k
    tempL_x = list(x_folds)
    tempDF_x_test = tempL_x.pop(model_k)
    tempA_x_train = np.concatenate(tempL_x)
    tempL_y = list(y_folds)
    tempDF_y_test = tempL_y.pop(model_k)
    tempA_y_train = np.concatenate(tempL_y)
    #Cross-validation with training dataset
    model.fit(tempA_x_train, tempA_y_train)
    #Save parameters (Skip intercepts and alphas here)
    metBMI_F_bcoefs['Model_'+str(model_k)] = list(model.coef_)#w in the cost function formula
    #Evaluation with testing dataset
    metBMI_F_scores.append(model.score(tempDF_x_test, tempDF_y_test))
    #Prediction with model #k for testing dataset
    tempL.append(model.predict(tempDF_x_test).flatten())
t_elapsed = time.time() - t_start
print('Elapsed time for 10 models of 10-fold CV LASSO:',
      round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean the predicted values
tempS = pd.Series([value for sublist in tempL for value in sublist],
                  index=tempDF.index, name='log_BaseMetBMI')
metBMI_F = pd.concat([tempDF, tempS], axis=1)
#Convert to original scale
metBMI_F['BaseBMI'] = np.e**metBMI_F['log_BaseBMI']
metBMI_F['BaseMetBMI'] = np.e**metBMI_F['log_BaseMetBMI']

display(metBMI_F)
#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_metBMI-Female.csv'
metBMI_F.to_csv(fileDir+fileName, index=True)

In [ ]:
#Male model
#Split into 10 datasets
x_folds = np.array_split(metDF_M_x, 10)
tempDF = metDF_M.drop(columns=list(set(metDF_M.columns)-set(['log_BaseBMI'])))#Not Series but DF
y_folds = np.array_split(tempDF, 10)

model = LassoCV(eps=0.05, n_alphas=200, alphas=None, fit_intercept=True,
                normalize=False, precompute='auto', cv=10)
metBMI_M_bcoefs = pd.DataFrame(index=metDF_M_x.columns).astype('float64')#For beta-coefficient
metBMI_M_bcoefs.index.set_names('Variable', inplace=True)
metBMI_M_scores = []#For the coefficient of determination R^2
tempL = []#For predictions
#Perform LASSO
t_start = time.time()
for model_k in range(10):
    #Set training/testing dataset in model #k
    tempL_x = list(x_folds)
    tempDF_x_test = tempL_x.pop(model_k)
    tempA_x_train = np.concatenate(tempL_x)
    tempL_y = list(y_folds)
    tempDF_y_test = tempL_y.pop(model_k)
    tempA_y_train = np.concatenate(tempL_y)
    #Cross-validation with training dataset
    model.fit(tempA_x_train, tempA_y_train)
    #Save parameters (Skip intercepts and alphas here)
    metBMI_M_bcoefs['Model_'+str(model_k)] = list(model.coef_)#w in the cost function formula
    #Evaluation with testing dataset
    metBMI_M_scores.append(model.score(tempDF_x_test, tempDF_y_test))
    #Prediction with model #k for testing dataset
    tempL.append(model.predict(tempDF_x_test).flatten())
t_elapsed = time.time() - t_start
print('Elapsed time for 10 models of 10-fold CV LASSO:',
      round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean the predicted values
tempS = pd.Series([value for sublist in tempL for value in sublist],
                  index=tempDF.index, name='log_BaseMetBMI')
metBMI_M = pd.concat([tempDF, tempS], axis=1)
#Convert to original scale
metBMI_M['BaseBMI'] = np.e**metBMI_M['log_BaseBMI']
metBMI_M['BaseMetBMI'] = np.e**metBMI_M['log_BaseMetBMI']

display(metBMI_M)
#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_metBMI-Male.csv'
metBMI_M.to_csv(fileDir+fileName, index=True)

In [ ]:
#Both sex model
#Split into 10 datasets
x_folds = np.array_split(metDF_B_x, 10)
tempDF = metDF_B.drop(columns=list(set(metDF_B.columns)-set(['log_BaseBMI'])))#Not Series but DF
y_folds = np.array_split(tempDF, 10)

model = LassoCV(eps=0.05, n_alphas=200, alphas=None, fit_intercept=True,
                normalize=False, precompute='auto', cv=10)
metBMI_B_bcoefs = pd.DataFrame(index=metDF_B_x.columns).astype('float64')#For beta-coefficient
metBMI_B_bcoefs.index.set_names('Variable', inplace=True)
metBMI_B_scores = []#For the coefficient of determination R^2
tempL = []#For predictions
#Perform LASSO
t_start = time.time()
for model_k in range(10):
    #Set training/testing dataset in model #k
    tempL_x = list(x_folds)
    tempDF_x_test = tempL_x.pop(model_k)
    tempA_x_train = np.concatenate(tempL_x)
    tempL_y = list(y_folds)
    tempDF_y_test = tempL_y.pop(model_k)
    tempA_y_train = np.concatenate(tempL_y)
    #Cross-validation with training dataset
    model.fit(tempA_x_train, tempA_y_train)
    #Save parameters (Skip intercepts and alphas here)
    metBMI_B_bcoefs['Model_'+str(model_k)] = list(model.coef_)#w in the cost function formula
    #Evaluation with testing dataset
    metBMI_B_scores.append(model.score(tempDF_x_test, tempDF_y_test))
    #Prediction with model #k for testing dataset
    tempL.append(model.predict(tempDF_x_test).flatten())
t_elapsed = time.time() - t_start
print('Elapsed time for 10 models of 10-fold CV LASSO:',
      round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean the predicted values
tempS = pd.Series([value for sublist in tempL for value in sublist],
                  index=tempDF.index, name='log_BaseMetBMI')
metBMI_B = pd.concat([tempDF, tempS], axis=1)
#Convert to original scale
metBMI_B['BaseBMI'] = np.e**metBMI_B['log_BaseBMI']
metBMI_B['BaseMetBMI'] = np.e**metBMI_B['log_BaseMetBMI']

display(metBMI_B)
#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_metBMI-BothSex.csv'
metBMI_B.to_csv(fileDir+fileName, index=True)

### 1-4. Prediction accuracy

In [ ]:
print('Female model')
print('• R2 Score in LASSO [Mean ± SD]:',
      np.mean(metBMI_F_scores), '±', np.std(metBMI_F_scores, ddof=1))#Sample standard deviation
print('• R2 Score in LASSO [Mean ± SEM]:',
      np.mean(metBMI_F_scores), '±', np.std(metBMI_F_scores, ddof=1)/np.sqrt(10))
print('• Observed vs. LASSO-predicted BMI (log) Pearson\'s r:',
      stats.pearsonr(metBMI_F['log_BaseBMI'], metBMI_F['log_BaseMetBMI']))#output: Pearson's r, p-value
print('• Observed vs. LASSO-predicted BMI Pearson\'s r:',
      stats.pearsonr(metBMI_F['BaseBMI'], metBMI_F['BaseMetBMI']))#output: Pearson's r, p-value
display(metBMI_F.describe())

print('Male model')
print('• R2 Score in LASSO [Mean ± SD]:',
      np.mean(metBMI_M_scores), '±', np.std(metBMI_M_scores, ddof=1))#Sample standard deviation
print('• R2 Score in LASSO [Mean ± SEM]:',
      np.mean(metBMI_M_scores), '±', np.std(metBMI_M_scores, ddof=1)/np.sqrt(10))
print('• Observed vs. LASSO-predicted BMI (log) Pearson\'s r:',
      stats.pearsonr(metBMI_M['log_BaseBMI'], metBMI_M['log_BaseMetBMI']))#output: Pearson's r, p-value
print('• Observed vs. LASSO-predicted BMI Pearson\'s r:',
      stats.pearsonr(metBMI_M['BaseBMI'], metBMI_M['BaseMetBMI']))#output: Pearson's r, p-value
display(metBMI_M.describe())

print('Both sex model')
print('• R2 Score in LASSO [Mean ± SD]:',
      np.mean(metBMI_B_scores), '±', np.std(metBMI_B_scores, ddof=1))#Sample standard deviation
print('• R2 Score in LASSO [Mean ± SEM]:',
      np.mean(metBMI_B_scores), '±', np.std(metBMI_B_scores, ddof=1)/np.sqrt(10))
print('• Observed vs. LASSO-predicted BMI (log) Pearson\'s r:',
      stats.pearsonr(metBMI_B['log_BaseBMI'], metBMI_B['log_BaseMetBMI']))#output: Pearson's r, p-value
print('• Observed vs. LASSO-predicted BMI Pearson\'s r:',
      stats.pearsonr(metBMI_B['BaseBMI'], metBMI_B['BaseMetBMI']))#output: Pearson's r, p-value
display(metBMI_B.describe())

In [ ]:
tempDF = pd.DataFrame({'Female':metBMI_F_scores, 'Male':metBMI_M_scores, 'Both sex':metBMI_B_scores})
tempDF = tempDF.melt(var_name='Cohort', value_name='R2score', value_vars=tempDF.columns.tolist())

#Plot
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(3, 1))
sns.barplot(data=tempDF, y='Cohort', x='R2score', hue='Cohort', palette='Set1', dodge=False, edgecolor='black',
            ci=68, capsize=0.4, errwidth=1.5, errcolor='black')#68%CI=Mean±SEM
sns.stripplot(data=tempDF, y='Cohort', x='R2score', hue='Cohort', dodge=False, size=8, edgecolor='black',
              linewidth=1, alpha=0.4, palette={'Female':'gray', 'Male':'gray', 'Both sex':'gray'})
sns.despine()
plt.xlabel('Out-of-sample '+r'$R^2$'+' in 10 LASSO models\n[mean ± SEM]')
plt.ylabel('')
plt.legend('', frameon=False)
plt.show()

In [ ]:
tempDF = pd.concat([metBMI_F, metBMI_M, metBMI_B], axis=0)
tempDF['Model'] = np.repeat(['Female', 'Male', 'Both sex'],
                            [len(metBMI_F), len(metBMI_M), len(metBMI_B)])

#Plot all models
sns.set(style='ticks', font='Arial', context='notebook')
sns.lmplot(data=tempDF, x='BaseMetBMI', y='BaseBMI', hue='Model', palette='Set1',
           scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=2.5, aspect=1)
plt.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))#Y=X as reference
plt.xlabel('Predicted BMI [kg m'+r'$^{-2}$'+']')
plt.ylabel('Measured BMI [kg m'+r'$^{-2}$'+']')
plt.show()

#Plot difference b/w sex
sns.set(style='ticks', font='Arial', context='talk')
sns.lmplot(data=tempDF[tempDF['Model']!='Both sex'], x='BaseMetBMI', y='BaseBMI',
           hue='Model', palette='Set1', scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=4
          ).set(xlim=(8, 57), xticks=np.arange(10, 51, 10), ylim=(8, 57), yticks=np.arange(10, 51, 10))
plt.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))#Y=X as reference
plt.xlabel('Predicted BMI [kg m'+r'$^{-2}$'+']')
plt.ylabel('Measured BMI [kg m'+r'$^{-2}$'+']')
plt.show()

#Plot only both-sex model
sns.set(style='ticks', font='Arial', context='talk')
sns.lmplot(data=metBMI_B, x='BaseMetBMI', y='BaseBMI', line_kws={'color':'b'},
           scatter_kws={'alpha':0.2, 'edgecolor':'black', 'color':'b'}, height=4
          ).set(xlim=(8, 57), xticks=np.arange(10, 51, 10), ylim=(8, 57), yticks=np.arange(10, 51, 10))
plt.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))#Y=X as reference
plt.xlabel('Predicted BMI [kg m'+r'$^{-2}$'+']')
plt.ylabel('Measured BMI [kg m'+r'$^{-2}$'+']')
plt.show()

#Plot difference b/w sex-combined and sex-stratified models
tempS = metDF_B['Sex']
tempDF = pd.merge(tempDF, tempS, left_index=True, right_index=True, how='left')
tempD = {'Female':'Sex-stratified', 'Male':'Sex-stratified', 'Both sex':'Sex-combined'}
tempDF['Model'] = tempDF['Model'].map(tempD)
tempDF = tempDF.reset_index().set_index(['public_client_id', 'Sex']).pivot(columns='Model')['BaseMetBMI']
tempDF = tempDF.reset_index()
tempD = {'F':'Female', 'M':'Male'}
sns.set(style='ticks', font='Arial', context='talk')
p = sns.lmplot(data=tempDF, x='Sex-stratified', y='Sex-combined',
               hue='Sex', hue_order=tempD.keys(), palette='Set1', legend=False,
               col='Sex', col_order=tempD.keys(), col_wrap=2,
               scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=4, aspect=1)
for ax, sex in zip(p.axes.ravel(), tempD.keys()):
    #Draw Y=X as reference
    ax.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))
    #Draw Pearson's r
    tempS1 = tempDF.loc[tempDF['Sex']==sex]['Sex-stratified']
    tempS2 = tempDF.loc[tempDF['Sex']==sex]['Sex-combined']
    pearson_r = stats.pearsonr(tempS1, tempS2)[0]
    ax.annotate('Pearson\'s '+r'$r$'+' = '+str(round(pearson_r, 3)), (10, 50), fontsize='small')
    #Change facet label
    ax.set_title(tempD[sex], {'fontsize':'large'})
p.set_axis_labels('Sex-stratified bBMI [kg m'+r'$^{-2}$'+']',
                  'Sex-combined bBMI [kg m'+r'$^{-2}$'+']')
plt.show()

### 1-5. Cleaning beta-coefficient dataframe

In [ ]:
#Female model
tempDF = metBMI_F_bcoefs.copy(deep=True)
print('Variables:', len(tempDF))

#Summarize
tempL1 = []
tempL2 = []
tempL3 = []
for row_n in tempDF.index.tolist():
    tempL1.append(tempDF.loc[row_n].mean())
    tempL2.append(tempDF.loc[row_n].std(ddof=1))#Sample standard deviation
    tempL3.append((tempDF.loc[row_n]==0.0).astype('int64').sum())
tempDF['MEAN'] = tempL1
tempDF['SD'] = tempL2
tempDF['nZeros'] = tempL3

#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_metBMI-Female_LASSO-bcoefs.tsv'
tempDF.to_csv(fileDir+fileName, sep='\t', index=True)

#Variables with non-zero beta-coefficient
tempDF = tempDF.loc[tempDF['nZeros']!=10]
print('Variables with non-zero beta-coefficient:', len(tempDF))

#Update
metBMI_F_bcoefs = tempDF

#Extract robust beta-coefficient: no zeros in all 10 models
tempDF = metBMI_F_bcoefs.loc[metBMI_F_bcoefs['nZeros']==0]
tempDF = tempDF.sort_values(by='MEAN', ascending=False)
print('Variables with non-zero beta-coefficient in all 10 models:', len(tempDF))
display(tempDF)

In [ ]:
#Male model
tempDF = metBMI_M_bcoefs.copy(deep=True)
print('Variables:', len(tempDF))

#Summarize
tempL1 = []
tempL2 = []
tempL3 = []
for row_n in tempDF.index.tolist():
    tempL1.append(tempDF.loc[row_n].mean())
    tempL2.append(tempDF.loc[row_n].std(ddof=1))#Sample standard deviation
    tempL3.append((tempDF.loc[row_n]==0.0).astype('int64').sum())
tempDF['MEAN'] = tempL1
tempDF['SD'] = tempL2
tempDF['nZeros'] = tempL3

#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_metBMI-Male_LASSO-bcoefs.tsv'
tempDF.to_csv(fileDir+fileName, sep='\t', index=True)

#Variables with non-zero beta-coefficient
tempDF = tempDF.loc[tempDF['nZeros']!=10]
print('Variables with non-zero beta-coefficient:', len(tempDF))

#Update
metBMI_M_bcoefs = tempDF

#Extract robust beta-coefficient: no zeros in all 10 models
tempDF = metBMI_M_bcoefs.loc[metBMI_M_bcoefs['nZeros']==0]
tempDF = tempDF.sort_values(by='MEAN', ascending=False)
print('Variables with non-zero beta-coefficient in all 10 models:', len(tempDF))
display(tempDF)

In [ ]:
#Both sex model
tempDF = metBMI_B_bcoefs.copy(deep=True)
print('Variables:', len(tempDF))

#Summarize
tempL1 = []
tempL2 = []
tempL3 = []
for row_n in tempDF.index.tolist():
    tempL1.append(tempDF.loc[row_n].mean())
    tempL2.append(tempDF.loc[row_n].std(ddof=1))#Sample standard deviation
    tempL3.append((tempDF.loc[row_n]==0.0).astype('int64').sum())
tempDF['MEAN'] = tempL1
tempDF['SD'] = tempL2
tempDF['nZeros'] = tempL3

#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_metBMI-BothSex_LASSO-bcoefs.tsv'
tempDF.to_csv(fileDir+fileName, sep='\t', index=True)

#Variables with non-zero beta-coefficient
tempDF = tempDF.loc[tempDF['nZeros']!=10]
print('Variables with non-zero beta-coefficient:', len(tempDF))

#Update
metBMI_B_bcoefs = tempDF

#Extract robust beta-coefficient: no zeros in all 10 models
tempDF = metBMI_B_bcoefs.loc[metBMI_B_bcoefs['nZeros']==0]
tempDF = tempDF.sort_values(by='MEAN', ascending=False)
print('Variables with non-zero beta-coefficient in all 10 models:', len(tempDF))
display(tempDF)

## 2. Clinical labs

### 2-1. Sex stratification

In [ ]:
#Import the cleaned baseline dataframe for LASSO
fileDir = './ExportData/'
fileName = '210104_Biological-BMI-paper_RF-imputation_baseline-chemDF-with-RF-imputation.tsv'
chemDF = pd.read_csv(fileDir+fileName, sep='\t', dtype={'public_client_id': str}).set_index('public_client_id')
display(chemDF)

In [ ]:
#Splitting dataset into sex-dependent cohorts
chemDF_F = chemDF.loc[chemDF['Sex']=='F']
chemDF_M = chemDF.loc[chemDF['Sex']=='M']
chemDF_B = chemDF#Not copy just rename
print('(Female, Male, Both): (',
      len(chemDF_F.index), ', ', len(chemDF_M.index), ', ', len(chemDF_B.index), ')')

### 2-2. Standarization

In [ ]:
#Female cohort
tempDF = chemDF_F.drop(columns=covarL)
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()

#Z-score transformation
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
tempA = scaler.fit_transform(tempDF)#Column direction
tempDF = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)

display(tempDF)
display(tempDF.describe())

#Confirmation
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

#Update
chemDF_F_x = tempDF

In [ ]:
#Male cohort
tempDF = chemDF_M.drop(columns=covarL)
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()

#Z-score transformation
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
tempA = scaler.fit_transform(tempDF)#Column direction
tempDF = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)

display(tempDF)
display(tempDF.describe())

#Confirmation
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

#Update
chemDF_M_x = tempDF

In [ ]:
#Both sex cohort
tempDF = chemDF_B.drop(columns=covarL)
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()

#Z-score transformation
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
tempA = scaler.fit_transform(tempDF)#Column direction
tempDF = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)

display(tempDF)
display(tempDF.describe())

#Confirmation
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

#Update
chemDF_B_x = tempDF

### 2-3. LASSO with cross-validation

In [ ]:
#Female model
#Split into 10 datasets
x_folds = np.array_split(chemDF_F_x, 10)
tempDF = chemDF_F.drop(columns=list(set(chemDF_F.columns)-set(['log_BaseBMI'])))#Not Series but DF
y_folds = np.array_split(tempDF, 10)

model = LassoCV(eps=0.05, n_alphas=200, alphas=None, fit_intercept=True,
                normalize=False, precompute='auto', cv=10)
chemBMI_F_bcoefs = pd.DataFrame(index=chemDF_F_x.columns).astype('float64')#For beta-coefficient
chemBMI_F_bcoefs.index.set_names('Variable', inplace=True)
chemBMI_F_scores = []#For the coefficient of determination R^2
tempL = []#For predictions
#Perform LASSO
t_start = time.time()
for model_k in range(10):
    #Set training/testing dataset in model #k
    tempL_x = list(x_folds)
    tempDF_x_test = tempL_x.pop(model_k)
    tempA_x_train = np.concatenate(tempL_x)
    tempL_y = list(y_folds)
    tempDF_y_test = tempL_y.pop(model_k)
    tempA_y_train = np.concatenate(tempL_y)
    #Cross-validation with training dataset
    model.fit(tempA_x_train, tempA_y_train)
    #Save parameters (Skip intercepts and alphas here)
    chemBMI_F_bcoefs['Model_'+str(model_k)] = list(model.coef_)#w in the cost function formula
    #Evaluation with testing dataset
    chemBMI_F_scores.append(model.score(tempDF_x_test, tempDF_y_test))
    #Prediction with model #k for testing dataset
    tempL.append(model.predict(tempDF_x_test).flatten())
t_elapsed = time.time() - t_start
print('Elapsed time for 10 models of 10-fold CV LASSO:',
      round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean the predicted values
tempS = pd.Series([value for sublist in tempL for value in sublist],
                  index=tempDF.index, name='log_BaseChemBMI')
chemBMI_F = pd.concat([tempDF, tempS], axis=1)
#Convert to original scale
chemBMI_F['BaseBMI'] = np.e**chemBMI_F['log_BaseBMI']
chemBMI_F['BaseChemBMI'] = np.e**chemBMI_F['log_BaseChemBMI']

display(chemBMI_F)
#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_chemBMI-Female.csv'
chemBMI_F.to_csv(fileDir+fileName, index=True)

In [ ]:
#Male model
#Split into 10 datasets
x_folds = np.array_split(chemDF_M_x, 10)
tempDF = chemDF_M.drop(columns=list(set(chemDF_M.columns)-set(['log_BaseBMI'])))#Not Series but DF
y_folds = np.array_split(tempDF, 10)

model = LassoCV(eps=0.05, n_alphas=200, alphas=None, fit_intercept=True,
                normalize=False, precompute='auto', cv=10)
chemBMI_M_bcoefs = pd.DataFrame(index=chemDF_M_x.columns).astype('float64')#For beta-coefficient
chemBMI_M_bcoefs.index.set_names('Variable', inplace=True)
chemBMI_M_scores = []#For the coefficient of determination R^2
tempL = []#For predictions
#Perform LASSO
t_start = time.time()
for model_k in range(10):
    #Set training/testing dataset in model #k
    tempL_x = list(x_folds)
    tempDF_x_test = tempL_x.pop(model_k)
    tempA_x_train = np.concatenate(tempL_x)
    tempL_y = list(y_folds)
    tempDF_y_test = tempL_y.pop(model_k)
    tempA_y_train = np.concatenate(tempL_y)
    #Cross-validation with training dataset
    model.fit(tempA_x_train, tempA_y_train)
    #Save parameters (Skip intercepts and alphas here)
    chemBMI_M_bcoefs['Model_'+str(model_k)] = list(model.coef_)#w in the cost function formula
    #Evaluation with testing dataset
    chemBMI_M_scores.append(model.score(tempDF_x_test, tempDF_y_test))
    #Prediction with model #k for testing dataset
    tempL.append(model.predict(tempDF_x_test).flatten())
t_elapsed = time.time() - t_start
print('Elapsed time for 10 models of 10-fold CV LASSO:',
      round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean the predicted values
tempS = pd.Series([value for sublist in tempL for value in sublist],
                  index=tempDF.index, name='log_BaseChemBMI')
chemBMI_M = pd.concat([tempDF, tempS], axis=1)
#Convert to original scale
chemBMI_M['BaseBMI'] = np.e**chemBMI_M['log_BaseBMI']
chemBMI_M['BaseChemBMI'] = np.e**chemBMI_M['log_BaseChemBMI']

display(chemBMI_M)
#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_chemBMI-Male.csv'
chemBMI_M.to_csv(fileDir+fileName, index=True)

In [ ]:
#Both sex model
#Split into 10 datasets
x_folds = np.array_split(chemDF_B_x, 10)
tempDF = chemDF_B.drop(columns=list(set(chemDF_B.columns)-set(['log_BaseBMI'])))#Not Series but DF
y_folds = np.array_split(tempDF, 10)

model = LassoCV(eps=0.05, n_alphas=200, alphas=None, fit_intercept=True,
                normalize=False, precompute='auto', cv=10)
chemBMI_B_bcoefs = pd.DataFrame(index=chemDF_B_x.columns).astype('float64')#For beta-coefficient
chemBMI_B_bcoefs.index.set_names('Variable', inplace=True)
chemBMI_B_scores = []#For the coefficient of determination R^2
tempL = []#For predictions
#Perform LASSO
t_start = time.time()
for model_k in range(10):
    #Set training/testing dataset in model #k
    tempL_x = list(x_folds)
    tempDF_x_test = tempL_x.pop(model_k)
    tempA_x_train = np.concatenate(tempL_x)
    tempL_y = list(y_folds)
    tempDF_y_test = tempL_y.pop(model_k)
    tempA_y_train = np.concatenate(tempL_y)
    #Cross-validation with training dataset
    model.fit(tempA_x_train, tempA_y_train)
    #Save parameters (Skip intercepts and alphas here)
    chemBMI_B_bcoefs['Model_'+str(model_k)] = list(model.coef_)#w in the cost function formula
    #Evaluation with testing dataset
    chemBMI_B_scores.append(model.score(tempDF_x_test, tempDF_y_test))
    #Prediction with model #k for testing dataset
    tempL.append(model.predict(tempDF_x_test).flatten())
t_elapsed = time.time() - t_start
print('Elapsed time for 10 models of 10-fold CV LASSO:',
      round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean the predicted values
tempS = pd.Series([value for sublist in tempL for value in sublist],
                  index=tempDF.index, name='log_BaseChemBMI')
chemBMI_B = pd.concat([tempDF, tempS], axis=1)
#Convert to original scale
chemBMI_B['BaseBMI'] = np.e**chemBMI_B['log_BaseBMI']
chemBMI_B['BaseChemBMI'] = np.e**chemBMI_B['log_BaseChemBMI']

display(chemBMI_B)
#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_chemBMI-BothSex.csv'
chemBMI_B.to_csv(fileDir+fileName, index=True)

### 2-4. Prediction accuracy

In [ ]:
print('Female model')
print('• R2 Score in LASSO [Mean ± SD]:',
      np.mean(chemBMI_F_scores), '±', np.std(chemBMI_F_scores, ddof=1))#Sample standard deviation
print('• R2 Score in LASSO [Mean ± SEM]:',
      np.mean(chemBMI_F_scores), '±', np.std(chemBMI_F_scores, ddof=1)/np.sqrt(10))
print('• Observed vs. LASSO-predicted BMI (log) Pearson\'s r:',
      stats.pearsonr(chemBMI_F['log_BaseBMI'], chemBMI_F['log_BaseChemBMI']))#output: Pearson's r, p-value
print('• Observed vs. LASSO-predicted BMI Pearson\'s r:',
      stats.pearsonr(chemBMI_F['BaseBMI'], chemBMI_F['BaseChemBMI']))#output: Pearson's r, p-value
display(chemBMI_F.describe())

print('Male model')
print('• R2 Score in LASSO [Mean ± SD]:',
      np.mean(chemBMI_M_scores), '±', np.std(chemBMI_M_scores, ddof=1))#Sample standard deviation
print('• R2 Score in LASSO [Mean ± SEM]:',
      np.mean(chemBMI_M_scores), '±', np.std(chemBMI_M_scores, ddof=1)/np.sqrt(10))
print('• Observed vs. LASSO-predicted BMI (log) Pearson\'s r:',
      stats.pearsonr(chemBMI_M['log_BaseBMI'], chemBMI_M['log_BaseChemBMI']))#output: Pearson's r, p-value
print('• Observed vs. LASSO-predicted BMI Pearson\'s r:',
      stats.pearsonr(chemBMI_M['BaseBMI'], chemBMI_M['BaseChemBMI']))#output: Pearson's r, p-value
display(chemBMI_M.describe())

print('Both sex model')
print('• R2 Score in LASSO [Mean ± SD]:',
      np.mean(chemBMI_B_scores), '±', np.std(chemBMI_B_scores, ddof=1))#Sample standard deviation
print('• R2 Score in LASSO [Mean ± SEM]:',
      np.mean(chemBMI_B_scores), '±', np.std(chemBMI_B_scores, ddof=1)/np.sqrt(10))
print('• Observed vs. LASSO-predicted BMI (log) Pearson\'s r:',
      stats.pearsonr(chemBMI_B['log_BaseBMI'], chemBMI_B['log_BaseChemBMI']))#output: Pearson's r, p-value
print('• Observed vs. LASSO-predicted BMI Pearson\'s r:',
      stats.pearsonr(chemBMI_B['BaseBMI'], chemBMI_B['BaseChemBMI']))#output: Pearson's r, p-value
display(chemBMI_B.describe())

In [ ]:
tempDF = pd.DataFrame({'Female':chemBMI_F_scores, 'Male':chemBMI_M_scores, 'Both sex':chemBMI_B_scores})
tempDF = tempDF.melt(var_name='Cohort', value_name='R2score', value_vars=tempDF.columns.tolist())

#Plot
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(3, 1))
sns.barplot(data=tempDF, y='Cohort', x='R2score', hue='Cohort', palette='Set1', dodge=False, edgecolor='black',
            ci=68, capsize=0.4, errwidth=1.5, errcolor='black')#68%CI=Mean±SEM
sns.stripplot(data=tempDF, y='Cohort', x='R2score', hue='Cohort', dodge=False, size=8, edgecolor='black',
              linewidth=1, alpha=0.4, palette={'Female':'gray', 'Male':'gray', 'Both sex':'gray'})
sns.despine()
plt.xlabel('Out-of-sample '+r'$R^2$'+' in 10 LASSO models\n[mean ± SEM]')
plt.ylabel('')
plt.legend('', frameon=False)
plt.show()

In [ ]:
tempDF = pd.concat([chemBMI_F, chemBMI_M, chemBMI_B], axis=0)
tempDF['Model'] = np.repeat(['Female', 'Male', 'Both sex'],
                            [len(chemBMI_F), len(chemBMI_M), len(chemBMI_B)])

#Plot all models
sns.set(style='ticks', font='Arial', context='notebook')
sns.lmplot(data=tempDF, x='BaseChemBMI', y='BaseBMI', hue='Model', palette='Set1',
           scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=2.5, aspect=1)
plt.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))#Y=X as reference
plt.xlabel('Predicted BMI [kg m'+r'$^{-2}$'+']')
plt.ylabel('Measured BMI [kg m'+r'$^{-2}$'+']')
plt.show()

#Plot difference b/w sex
sns.set(style='ticks', font='Arial', context='talk')
sns.lmplot(data=tempDF[tempDF['Model']!='Both sex'], x='BaseChemBMI', y='BaseBMI',
           hue='Model', palette='Set1', scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=4
          ).set(xlim=(8, 57), xticks=np.arange(10, 51, 10), ylim=(8, 57), yticks=np.arange(10, 51, 10))
plt.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))#Y=X as reference
plt.xlabel('Predicted BMI [kg m'+r'$^{-2}$'+']')
plt.ylabel('Measured BMI [kg m'+r'$^{-2}$'+']')
plt.show()

#Plot only both-sex model
sns.set(style='ticks', font='Arial', context='talk')
sns.lmplot(data=chemBMI_B, x='BaseChemBMI', y='BaseBMI', line_kws={'color':'g'},
           scatter_kws={'alpha':0.2, 'edgecolor':'black', 'color':'g'}, height=4
          ).set(xlim=(8, 57), xticks=np.arange(10, 51, 10), ylim=(8, 57), yticks=np.arange(10, 51, 10))
plt.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))#Y=X as reference
plt.xlabel('Predicted BMI [kg m'+r'$^{-2}$'+']')
plt.ylabel('Measured BMI [kg m'+r'$^{-2}$'+']')
plt.show()

#Plot difference b/w sex-combined and sex-stratified models
tempS = chemDF_B['Sex']
tempDF = pd.merge(tempDF, tempS, left_index=True, right_index=True, how='left')
tempD = {'Female':'Sex-stratified', 'Male':'Sex-stratified', 'Both sex':'Sex-combined'}
tempDF['Model'] = tempDF['Model'].map(tempD)
tempDF = tempDF.reset_index().set_index(['public_client_id', 'Sex']).pivot(columns='Model')['BaseChemBMI']
tempDF = tempDF.reset_index()
tempD = {'F':'Female', 'M':'Male'}
sns.set(style='ticks', font='Arial', context='talk')
p = sns.lmplot(data=tempDF, x='Sex-stratified', y='Sex-combined',
               hue='Sex', hue_order=tempD.keys(), palette='Set1', legend=False,
               col='Sex', col_order=tempD.keys(), col_wrap=2,
               scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=4, aspect=1)
for ax, sex in zip(p.axes.ravel(), tempD.keys()):
    #Draw Y=X as reference
    ax.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))
    #Draw Pearson's r
    tempS1 = tempDF.loc[tempDF['Sex']==sex]['Sex-stratified']
    tempS2 = tempDF.loc[tempDF['Sex']==sex]['Sex-combined']
    pearson_r = stats.pearsonr(tempS1, tempS2)[0]
    ax.annotate('Pearson\'s '+r'$r$'+' = '+str(round(pearson_r, 3)), (10, 50), fontsize='small')
    #Change facet label
    ax.set_title(tempD[sex], {'fontsize':'large'})
p.set_axis_labels('Sex-stratified bBMI [kg m'+r'$^{-2}$'+']',
                  'Sex-combined bBMI [kg m'+r'$^{-2}$'+']')
plt.show()

### 2-5. Cleaning beta-coefficient dataframe

In [ ]:
#Female model
tempDF = chemBMI_F_bcoefs.copy(deep=True)
print('Variables:', len(tempDF))

#Summarize
tempL1 = []
tempL2 = []
tempL3 = []
for row_n in tempDF.index.tolist():
    tempL1.append(tempDF.loc[row_n].mean())
    tempL2.append(tempDF.loc[row_n].std(ddof=1))#Sample standard deviation
    tempL3.append((tempDF.loc[row_n]==0.0).astype('int64').sum())
tempDF['MEAN'] = tempL1
tempDF['SD'] = tempL2
tempDF['nZeros'] = tempL3

#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_chemBMI-Female_LASSO-bcoefs.tsv'
tempDF.to_csv(fileDir+fileName, sep='\t', index=True)

#Variables with non-zero beta-coefficient
tempDF = tempDF.loc[tempDF['nZeros']!=10]
print('Variables with non-zero beta-coefficient:', len(tempDF))

#Update
chemBMI_F_bcoefs = tempDF

#Extract robust beta-coefficient: no zeros in all 10 models
tempDF = chemBMI_F_bcoefs.loc[chemBMI_F_bcoefs['nZeros']==0]
tempDF = tempDF.sort_values(by='MEAN', ascending=False)
print('Variables with non-zero beta-coefficient in all 10 models:', len(tempDF))
display(tempDF)

In [ ]:
#Male model
tempDF = chemBMI_M_bcoefs.copy(deep=True)
print('Variables:', len(tempDF))

#Summarize
tempL1 = []
tempL2 = []
tempL3 = []
for row_n in tempDF.index.tolist():
    tempL1.append(tempDF.loc[row_n].mean())
    tempL2.append(tempDF.loc[row_n].std(ddof=1))#Sample standard deviation
    tempL3.append((tempDF.loc[row_n]==0.0).astype('int64').sum())
tempDF['MEAN'] = tempL1
tempDF['SD'] = tempL2
tempDF['nZeros'] = tempL3

#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_chemBMI-Male_LASSO-bcoefs.tsv'
tempDF.to_csv(fileDir+fileName, sep='\t', index=True)

#Variables with non-zero beta-coefficient
tempDF = tempDF.loc[tempDF['nZeros']!=10]
print('Variables with non-zero beta-coefficient:', len(tempDF))

#Update
chemBMI_M_bcoefs = tempDF

#Extract robust beta-coefficient: no zeros in all 10 models
tempDF = chemBMI_M_bcoefs.loc[chemBMI_M_bcoefs['nZeros']==0]
tempDF = tempDF.sort_values(by='MEAN', ascending=False)
print('Variables with non-zero beta-coefficient in all 10 models:', len(tempDF))
display(tempDF)

In [ ]:
#Both sex model
tempDF = chemBMI_B_bcoefs.copy(deep=True)
print('Variables:', len(tempDF))

#Summarize
tempL1 = []
tempL2 = []
tempL3 = []
for row_n in tempDF.index.tolist():
    tempL1.append(tempDF.loc[row_n].mean())
    tempL2.append(tempDF.loc[row_n].std(ddof=1))#Sample standard deviation
    tempL3.append((tempDF.loc[row_n]==0.0).astype('int64').sum())
tempDF['MEAN'] = tempL1
tempDF['SD'] = tempL2
tempDF['nZeros'] = tempL3

#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_chemBMI-BothSex_LASSO-bcoefs.tsv'
tempDF.to_csv(fileDir+fileName, sep='\t', index=True)

#Variables with non-zero beta-coefficient
tempDF = tempDF.loc[tempDF['nZeros']!=10]
print('Variables with non-zero beta-coefficient:', len(tempDF))

#Update
chemBMI_B_bcoefs = tempDF

#Extract robust beta-coefficient: no zeros in all 10 models
tempDF = chemBMI_B_bcoefs.loc[chemBMI_B_bcoefs['nZeros']==0]
tempDF = tempDF.sort_values(by='MEAN', ascending=False)
print('Variables with non-zero beta-coefficient in all 10 models:', len(tempDF))
display(tempDF)

## 3. Proteomics

### 3-1. Sex stratification

In [ ]:
#Import the cleaned baseline dataframe for LASSO
fileDir = './ExportData/'
fileName = '210104_Biological-BMI-paper_RF-imputation_baseline-protDF-with-RF-imputation.tsv'
protDF = pd.read_csv(fileDir+fileName, sep='\t', dtype={'public_client_id': str}).set_index('public_client_id')
display(protDF)

In [ ]:
#Splitting dataset into sex-dependent cohorts
protDF_F = protDF.loc[protDF['Sex']=='F']
protDF_M = protDF.loc[protDF['Sex']=='M']
protDF_B = protDF#Not copy just rename
print('(Female, Male, Both): (',
      len(protDF_F.index), ', ', len(protDF_M.index), ', ', len(protDF_B.index), ')')

### 3-2. Standarization

In [ ]:
#Female cohort
tempDF = protDF_F.drop(columns=covarL)
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()

#Z-score transformation
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
tempA = scaler.fit_transform(tempDF)#Column direction
tempDF = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)

display(tempDF)
display(tempDF.describe())

#Confirmation
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

#Update
protDF_F_x = tempDF

In [ ]:
#Male cohort
tempDF = protDF_M.drop(columns=covarL)
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()

#Z-score transformation
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
tempA = scaler.fit_transform(tempDF)#Column direction
tempDF = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)

display(tempDF)
display(tempDF.describe())

#Confirmation
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

#Update
protDF_M_x = tempDF

In [ ]:
#Both sex cohort
tempDF = protDF_B.drop(columns=covarL)
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()

#Z-score transformation
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
tempA = scaler.fit_transform(tempDF)#Column direction
tempDF = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)

display(tempDF)
display(tempDF.describe())

#Confirmation
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

#Update
protDF_B_x = tempDF

### 3-3. LASSO with cross-validation

In [ ]:
#Female model
#Split into 10 datasets
x_folds = np.array_split(protDF_F_x, 10)
tempDF = protDF_F.drop(columns=list(set(protDF_F.columns)-set(['log_BaseBMI'])))#Not Series but DF
y_folds = np.array_split(tempDF, 10)

model = LassoCV(eps=0.05, n_alphas=200, alphas=None, fit_intercept=True,
                normalize=False, precompute='auto', cv=10)
protBMI_F_bcoefs = pd.DataFrame(index=protDF_F_x.columns).astype('float64')#For beta-coefficient
protBMI_F_bcoefs.index.set_names('Variable', inplace=True)
protBMI_F_scores = []#For the coefficient of determination R^2
tempL = []#For predictions
#Perform LASSO
t_start = time.time()
for model_k in range(10):
    #Set training/testing dataset in model #k
    tempL_x = list(x_folds)
    tempDF_x_test = tempL_x.pop(model_k)
    tempA_x_train = np.concatenate(tempL_x)
    tempL_y = list(y_folds)
    tempDF_y_test = tempL_y.pop(model_k)
    tempA_y_train = np.concatenate(tempL_y)
    #Cross-validation with training dataset
    model.fit(tempA_x_train, tempA_y_train)
    #Save parameters (Skip intercepts and alphas here)
    protBMI_F_bcoefs['Model_'+str(model_k)] = list(model.coef_)#w in the cost function formula
    #Evaluation with testing dataset
    protBMI_F_scores.append(model.score(tempDF_x_test, tempDF_y_test))
    #Prediction with model #k for testing dataset
    tempL.append(model.predict(tempDF_x_test).flatten())
t_elapsed = time.time() - t_start
print('Elapsed time for 10 models of 10-fold CV LASSO:',
      round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean the predicted values
tempS = pd.Series([value for sublist in tempL for value in sublist],
                  index=tempDF.index, name='log_BaseProtBMI')
protBMI_F = pd.concat([tempDF, tempS], axis=1)
#Convert to original scale
protBMI_F['BaseBMI'] = np.e**protBMI_F['log_BaseBMI']
protBMI_F['BaseProtBMI'] = np.e**protBMI_F['log_BaseProtBMI']

display(protBMI_F)
#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_protBMI-Female.csv'
protBMI_F.to_csv(fileDir+fileName, index=True)

In [ ]:
#Male model
#Split into 10 datasets
x_folds = np.array_split(protDF_M_x, 10)
tempDF = protDF_M.drop(columns=list(set(protDF_M.columns)-set(['log_BaseBMI'])))#Not Series but DF
y_folds = np.array_split(tempDF, 10)

model = LassoCV(eps=0.05, n_alphas=200, alphas=None, fit_intercept=True,
                normalize=False, precompute='auto', cv=10)
protBMI_M_bcoefs = pd.DataFrame(index=protDF_M_x.columns).astype('float64')#For beta-coefficient
protBMI_M_bcoefs.index.set_names('Variable', inplace=True)
protBMI_M_scores = []#For the coefficient of determination R^2
tempL = []#For predictions
#Perform LASSO
t_start = time.time()
for model_k in range(10):
    #Set training/testing dataset in model #k
    tempL_x = list(x_folds)
    tempDF_x_test = tempL_x.pop(model_k)
    tempA_x_train = np.concatenate(tempL_x)
    tempL_y = list(y_folds)
    tempDF_y_test = tempL_y.pop(model_k)
    tempA_y_train = np.concatenate(tempL_y)
    #Cross-validation with training dataset
    model.fit(tempA_x_train, tempA_y_train)
    #Save parameters (Skip intercepts and alphas here)
    protBMI_M_bcoefs['Model_'+str(model_k)] = list(model.coef_)#w in the cost function formula
    #Evaluation with testing dataset
    protBMI_M_scores.append(model.score(tempDF_x_test, tempDF_y_test))
    #Prediction with model #k for testing dataset
    tempL.append(model.predict(tempDF_x_test).flatten())
t_elapsed = time.time() - t_start
print('Elapsed time for 10 models of 10-fold CV LASSO:',
      round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean the predicted values
tempS = pd.Series([value for sublist in tempL for value in sublist],
                  index=tempDF.index, name='log_BaseProtBMI')
protBMI_M = pd.concat([tempDF, tempS], axis=1)
#Convert to original scale
protBMI_M['BaseBMI'] = np.e**protBMI_M['log_BaseBMI']
protBMI_M['BaseProtBMI'] = np.e**protBMI_M['log_BaseProtBMI']

display(protBMI_M)
#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_protBMI-Male.csv'
protBMI_M.to_csv(fileDir+fileName, index=True)

In [ ]:
#Both sex model
#Split into 10 datasets
x_folds = np.array_split(protDF_B_x, 10)
tempDF = protDF_B.drop(columns=list(set(protDF_B.columns)-set(['log_BaseBMI'])))#Not Series but DF
y_folds = np.array_split(tempDF, 10)

model = LassoCV(eps=0.05, n_alphas=200, alphas=None, fit_intercept=True,
                normalize=False, precompute='auto', cv=10)
protBMI_B_bcoefs = pd.DataFrame(index=protDF_B_x.columns).astype('float64')#For beta-coefficient
protBMI_B_bcoefs.index.set_names('Variable', inplace=True)
protBMI_B_scores = []#For the coefficient of determination R^2
tempL = []#For predictions
#Perform LASSO
t_start = time.time()
for model_k in range(10):
    #Set training/testing dataset in model #k
    tempL_x = list(x_folds)
    tempDF_x_test = tempL_x.pop(model_k)
    tempA_x_train = np.concatenate(tempL_x)
    tempL_y = list(y_folds)
    tempDF_y_test = tempL_y.pop(model_k)
    tempA_y_train = np.concatenate(tempL_y)
    #Cross-validation with training dataset
    model.fit(tempA_x_train, tempA_y_train)
    #Save parameters (Skip intercepts and alphas here)
    protBMI_B_bcoefs['Model_'+str(model_k)] = list(model.coef_)#w in the cost function formula
    #Evaluation with testing dataset
    protBMI_B_scores.append(model.score(tempDF_x_test, tempDF_y_test))
    #Prediction with model #k for testing dataset
    tempL.append(model.predict(tempDF_x_test).flatten())
t_elapsed = time.time() - t_start
print('Elapsed time for 10 models of 10-fold CV LASSO:',
      round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean the predicted values
tempS = pd.Series([value for sublist in tempL for value in sublist],
                  index=tempDF.index, name='log_BaseProtBMI')
protBMI_B = pd.concat([tempDF, tempS], axis=1)
#Convert to original scale
protBMI_B['BaseBMI'] = np.e**protBMI_B['log_BaseBMI']
protBMI_B['BaseProtBMI'] = np.e**protBMI_B['log_BaseProtBMI']

display(protBMI_B)
#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_protBMI-BothSex.csv'
protBMI_B.to_csv(fileDir+fileName, index=True)

### 3-4. Prediction accuracy

In [ ]:
print('Female model')
print('• R2 Score in LASSO [Mean ± SD]:',
      np.mean(protBMI_F_scores), '±', np.std(protBMI_F_scores, ddof=1))#Sample standard deviation
print('• R2 Score in LASSO [Mean ± SEM]:',
      np.mean(protBMI_F_scores), '±', np.std(protBMI_F_scores, ddof=1)/np.sqrt(10))
print('• Observed vs. LASSO-predicted BMI (log) Pearson\'s r:',
      stats.pearsonr(protBMI_F['log_BaseBMI'], protBMI_F['log_BaseProtBMI']))#output: Pearson's r, p-value
print('• Observed vs. LASSO-predicted BMI Pearson\'s r:',
      stats.pearsonr(protBMI_F['BaseBMI'], protBMI_F['BaseProtBMI']))#output: Pearson's r, p-value
display(protBMI_F.describe())

print('Male model')
print('• R2 Score in LASSO [Mean ± SD]:',
      np.mean(protBMI_M_scores), '±', np.std(protBMI_M_scores, ddof=1))#Sample standard deviation
print('• R2 Score in LASSO [Mean ± SEM]:',
      np.mean(protBMI_M_scores), '±', np.std(protBMI_M_scores, ddof=1)/np.sqrt(10))
print('• Observed vs. LASSO-predicted BMI (log) Pearson\'s r:',
      stats.pearsonr(protBMI_M['log_BaseBMI'], protBMI_M['log_BaseProtBMI']))#output: Pearson's r, p-value
print('• Observed vs. LASSO-predicted BMI Pearson\'s r:',
      stats.pearsonr(protBMI_M['BaseBMI'], protBMI_M['BaseProtBMI']))#output: Pearson's r, p-value
display(protBMI_M.describe())

print('Both sex model')
print('• R2 Score in LASSO [Mean ± SD]:',
      np.mean(protBMI_B_scores), '±', np.std(protBMI_B_scores, ddof=1))#Sample standard deviation
print('• R2 Score in LASSO [Mean ± SEM]:',
      np.mean(protBMI_B_scores), '±', np.std(protBMI_B_scores, ddof=1)/np.sqrt(10))
print('• Observed vs. LASSO-predicted BMI (log) Pearson\'s r:',
      stats.pearsonr(protBMI_B['log_BaseBMI'], protBMI_B['log_BaseProtBMI']))#output: Pearson's r, p-value
print('• Observed vs. LASSO-predicted BMI Pearson\'s r:',
      stats.pearsonr(protBMI_B['BaseBMI'], protBMI_B['BaseProtBMI']))#output: Pearson's r, p-value
display(protBMI_B.describe())

In [ ]:
tempDF = pd.DataFrame({'Female':protBMI_F_scores, 'Male':protBMI_M_scores, 'Both sex':protBMI_B_scores})
tempDF = tempDF.melt(var_name='Cohort', value_name='R2score', value_vars=tempDF.columns.tolist())

#Plot
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(3, 1))
sns.barplot(data=tempDF, y='Cohort', x='R2score', hue='Cohort', palette='Set1', dodge=False, edgecolor='black',
            ci=68, capsize=0.4, errwidth=1.5, errcolor='black')#68%CI=Mean±SEM
sns.stripplot(data=tempDF, y='Cohort', x='R2score', hue='Cohort', dodge=False, size=8, edgecolor='black',
              linewidth=1, alpha=0.4, palette={'Female':'gray', 'Male':'gray', 'Both sex':'gray'})
sns.despine()
plt.xlabel('Out-of-sample '+r'$R^2$'+' in 10 LASSO models\n[mean ± SEM]')
plt.ylabel('')
plt.legend('', frameon=False)
plt.show()

In [ ]:
tempDF = pd.concat([protBMI_F, protBMI_M, protBMI_B], axis=0)
tempDF['Model'] = np.repeat(['Female', 'Male', 'Both sex'],
                            [len(protBMI_F), len(protBMI_M), len(protBMI_B)])

#Plot all models
sns.set(style='ticks', font='Arial', context='notebook')
sns.lmplot(data=tempDF, x='BaseProtBMI', y='BaseBMI', hue='Model', palette='Set1',
           scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=2.5, aspect=1)
plt.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))#Y=X as reference
plt.xlabel('Predicted BMI [kg m'+r'$^{-2}$'+']')
plt.ylabel('Measured BMI [kg m'+r'$^{-2}$'+']')
plt.show()

#Plot difference b/w sex
sns.set(style='ticks', font='Arial', context='talk')
sns.lmplot(data=tempDF[tempDF['Model']!='Both sex'], x='BaseProtBMI', y='BaseBMI',
           hue='Model', palette='Set1', scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=4
          ).set(xlim=(8, 57), xticks=np.arange(10, 51, 10), ylim=(8, 57), yticks=np.arange(10, 51, 10))
plt.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))#Y=X as reference
plt.xlabel('Predicted BMI [kg m'+r'$^{-2}$'+']')
plt.ylabel('Measured BMI [kg m'+r'$^{-2}$'+']')
plt.show()

#Plot only both-sex model
sns.set(style='ticks', font='Arial', context='talk')
sns.lmplot(data=protBMI_B, x='BaseProtBMI', y='BaseBMI', line_kws={'color':'r'},
           scatter_kws={'alpha':0.2, 'edgecolor':'black', 'color':'r'}, height=4
          ).set(xlim=(8, 57), xticks=np.arange(10, 51, 10), ylim=(8, 57), yticks=np.arange(10, 51, 10))
plt.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))#Y=X as reference
plt.xlabel('Predicted BMI [kg m'+r'$^{-2}$'+']')
plt.ylabel('Measured BMI [kg m'+r'$^{-2}$'+']')
plt.show()

#Plot difference b/w sex-combined and sex-stratified models
tempS = protDF_B['Sex']
tempDF = pd.merge(tempDF, tempS, left_index=True, right_index=True, how='left')
tempD = {'Female':'Sex-stratified', 'Male':'Sex-stratified', 'Both sex':'Sex-combined'}
tempDF['Model'] = tempDF['Model'].map(tempD)
tempDF = tempDF.reset_index().set_index(['public_client_id', 'Sex']).pivot(columns='Model')['BaseProtBMI']
tempDF = tempDF.reset_index()
tempD = {'F':'Female', 'M':'Male'}
sns.set(style='ticks', font='Arial', context='talk')
p = sns.lmplot(data=tempDF, x='Sex-stratified', y='Sex-combined',
               hue='Sex', hue_order=tempD.keys(), palette='Set1', legend=False,
               col='Sex', col_order=tempD.keys(), col_wrap=2,
               scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=4, aspect=1)
for ax, sex in zip(p.axes.ravel(), tempD.keys()):
    #Draw Y=X as reference
    ax.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))
    #Draw Pearson's r
    tempS1 = tempDF.loc[tempDF['Sex']==sex]['Sex-stratified']
    tempS2 = tempDF.loc[tempDF['Sex']==sex]['Sex-combined']
    pearson_r = stats.pearsonr(tempS1, tempS2)[0]
    ax.annotate('Pearson\'s '+r'$r$'+' = '+str(round(pearson_r, 3)), (10, 50), fontsize='small')
    #Change facet label
    ax.set_title(tempD[sex], {'fontsize':'large'})
p.set_axis_labels('Sex-stratified bBMI [kg m'+r'$^{-2}$'+']',
                  'Sex-combined bBMI [kg m'+r'$^{-2}$'+']')
plt.show()

### 3-5. Cleaning beta-coefficient dataframe

In [ ]:
#Female model
tempDF = protBMI_F_bcoefs.copy(deep=True)
print('Variables:', len(tempDF))

#Summarize
tempL1 = []
tempL2 = []
tempL3 = []
for row_n in tempDF.index.tolist():
    tempL1.append(tempDF.loc[row_n].mean())
    tempL2.append(tempDF.loc[row_n].std(ddof=1))#Sample standard deviation
    tempL3.append((tempDF.loc[row_n]==0.0).astype('int64').sum())
tempDF['MEAN'] = tempL1
tempDF['SD'] = tempL2
tempDF['nZeros'] = tempL3

#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_protBMI-Female_LASSO-bcoefs.tsv'
tempDF.to_csv(fileDir+fileName, sep='\t', index=True)

#Variables with non-zero beta-coefficient
tempDF = tempDF.loc[tempDF['nZeros']!=10]
print('Variables with non-zero beta-coefficient:', len(tempDF))

#Update
protBMI_F_bcoefs = tempDF

#Extract robust beta-coefficient: no zeros in all 10 models
tempDF = protBMI_F_bcoefs.loc[protBMI_F_bcoefs['nZeros']==0]
tempDF = tempDF.sort_values(by='MEAN', ascending=False)
print('Variables with non-zero beta-coefficient in all 10 models:', len(tempDF))
display(tempDF)

In [ ]:
#Male model
tempDF = protBMI_M_bcoefs.copy(deep=True)
print('Variables:', len(tempDF))

#Summarize
tempL1 = []
tempL2 = []
tempL3 = []
for row_n in tempDF.index.tolist():
    tempL1.append(tempDF.loc[row_n].mean())
    tempL2.append(tempDF.loc[row_n].std(ddof=1))#Sample standard deviation
    tempL3.append((tempDF.loc[row_n]==0.0).astype('int64').sum())
tempDF['MEAN'] = tempL1
tempDF['SD'] = tempL2
tempDF['nZeros'] = tempL3

#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_protBMI-Male_LASSO-bcoefs.tsv'
tempDF.to_csv(fileDir+fileName, sep='\t', index=True)

#Variables with non-zero beta-coefficient
tempDF = tempDF.loc[tempDF['nZeros']!=10]
print('Variables with non-zero beta-coefficient:', len(tempDF))

#Update
protBMI_M_bcoefs = tempDF

#Extract robust beta-coefficient: no zeros in all 10 models
tempDF = protBMI_M_bcoefs.loc[protBMI_M_bcoefs['nZeros']==0]
tempDF = tempDF.sort_values(by='MEAN', ascending=False)
print('Variables with non-zero beta-coefficient in all 10 models:', len(tempDF))
display(tempDF)

In [ ]:
#Both sex model
tempDF = protBMI_B_bcoefs.copy(deep=True)
print('Variables:', len(tempDF))

#Summarize
tempL1 = []
tempL2 = []
tempL3 = []
for row_n in tempDF.index.tolist():
    tempL1.append(tempDF.loc[row_n].mean())
    tempL2.append(tempDF.loc[row_n].std(ddof=1))#Sample standard deviation
    tempL3.append((tempDF.loc[row_n]==0.0).astype('int64').sum())
tempDF['MEAN'] = tempL1
tempDF['SD'] = tempL2
tempDF['nZeros'] = tempL3

#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_protBMI-BothSex_LASSO-bcoefs.tsv'
tempDF.to_csv(fileDir+fileName, sep='\t', index=True)

#Variables with non-zero beta-coefficient
tempDF = tempDF.loc[tempDF['nZeros']!=10]
print('Variables with non-zero beta-coefficient:', len(tempDF))

#Update
protBMI_B_bcoefs = tempDF

#Extract robust beta-coefficient: no zeros in all 10 models
tempDF = protBMI_B_bcoefs.loc[protBMI_B_bcoefs['nZeros']==0]
tempDF = tempDF.sort_values(by='MEAN', ascending=False)
print('Variables with non-zero beta-coefficient in all 10 models:', len(tempDF))
display(tempDF)

## 4. Metabolomics, Clinical labs, Proteomics-combined omics

### 4-1. Sex stratification

In [ ]:
#Import the cleaned baseline dataframe for LASSO
fileDir = './ExportData/'
fileName = '210104_Biological-BMI-paper_RF-imputation_baseline-combiDF-with-RF-imputation.tsv'
combiDF = pd.read_csv(fileDir+fileName, sep='\t', dtype={'public_client_id': str}).set_index('public_client_id')
display(combiDF)

In [ ]:
#Splitting dataset into sex-dependent cohorts
combiDF_F = combiDF.loc[combiDF['Sex']=='F']
combiDF_M = combiDF.loc[combiDF['Sex']=='M']
combiDF_B = combiDF#Not copy just rename
print('(Female, Male, Both): (',
      len(combiDF_F.index), ', ', len(combiDF_M.index), ', ', len(combiDF_B.index), ')')

### 4-2. Standarization

In [ ]:
#Female cohort
tempDF = combiDF_F.drop(columns=covarL)
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()

#Z-score transformation
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
tempA = scaler.fit_transform(tempDF)#Column direction
tempDF = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)

display(tempDF)
display(tempDF.describe())

#Confirmation
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

#Update
combiDF_F_x = tempDF

In [ ]:
#Male cohort
tempDF = combiDF_M.drop(columns=covarL)
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()

#Z-score transformation
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
tempA = scaler.fit_transform(tempDF)#Column direction
tempDF = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)

display(tempDF)
display(tempDF.describe())

#Confirmation
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

#Update
combiDF_M_x = tempDF

In [ ]:
#Both sex cohort
tempDF = combiDF_B.drop(columns=covarL)
tempL1 = tempDF.index.tolist()
tempL2 = tempDF.columns.tolist()

#Z-score transformation
scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
tempA = scaler.fit_transform(tempDF)#Column direction
tempDF = pd.DataFrame(data=tempA, index=tempL1, columns=tempL2)

display(tempDF)
display(tempDF.describe())

#Confirmation
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for col_i in range(3):
    sns.distplot(tempDF.iloc[:, col_i], label=tempDF.columns[col_i])
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

#Update
combiDF_B_x = tempDF

### 4-3. LASSO with cross-validation

In [ ]:
#Female model
#Split into 10 datasets
x_folds = np.array_split(combiDF_F_x, 10)
tempDF = combiDF_F.drop(columns=list(set(combiDF_F.columns)-set(['log_BaseBMI'])))#Not Series but DF
y_folds = np.array_split(tempDF, 10)

model = LassoCV(eps=0.01, n_alphas=200, alphas=None, fit_intercept=True,
                normalize=False, precompute='auto', cv=10)
combiBMI_F_bcoefs = pd.DataFrame(index=combiDF_F_x.columns).astype('float64')#For beta-coefficient
combiBMI_F_bcoefs.index.set_names('Variable', inplace=True)
combiBMI_F_scores = []#For the coefficient of determination R^2
tempL = []#For predictions
#Perform LASSO
t_start = time.time()
for model_k in range(10):
    #Set training/testing dataset in model #k
    tempL_x = list(x_folds)
    tempDF_x_test = tempL_x.pop(model_k)
    tempA_x_train = np.concatenate(tempL_x)
    tempL_y = list(y_folds)
    tempDF_y_test = tempL_y.pop(model_k)
    tempA_y_train = np.concatenate(tempL_y)
    #Cross-validation with training dataset
    model.fit(tempA_x_train, tempA_y_train)
    #Save parameters (Skip intercepts and alphas here)
    combiBMI_F_bcoefs['Model_'+str(model_k)] = list(model.coef_)#w in the cost function formula
    #Evaluation with testing dataset
    combiBMI_F_scores.append(model.score(tempDF_x_test, tempDF_y_test))
    #Prediction with model #k for testing dataset
    tempL.append(model.predict(tempDF_x_test).flatten())
t_elapsed = time.time() - t_start
print('Elapsed time for 10 models of 10-fold CV LASSO:',
      round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean the predicted values
tempS = pd.Series([value for sublist in tempL for value in sublist],
                  index=tempDF.index, name='log_BaseCombiBMI')
combiBMI_F = pd.concat([tempDF, tempS], axis=1)
#Convert to original scale
combiBMI_F['BaseBMI'] = np.e**combiBMI_F['log_BaseBMI']
combiBMI_F['BaseCombiBMI'] = np.e**combiBMI_F['log_BaseCombiBMI']

display(combiBMI_F)
#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_combiBMI-Female.csv'
combiBMI_F.to_csv(fileDir+fileName, index=True)

In [ ]:
#Male model
#Split into 10 datasets
x_folds = np.array_split(combiDF_M_x, 10)
tempDF = combiDF_M.drop(columns=list(set(combiDF_M.columns)-set(['log_BaseBMI'])))#Not Series but DF
y_folds = np.array_split(tempDF, 10)

model = LassoCV(eps=0.01, n_alphas=200, alphas=None, fit_intercept=True,
                normalize=False, precompute='auto', cv=10)
combiBMI_M_bcoefs = pd.DataFrame(index=combiDF_M_x.columns).astype('float64')#For beta-coefficient
combiBMI_M_bcoefs.index.set_names('Variable', inplace=True)
combiBMI_M_scores = []#For the coefficient of determination R^2
tempL = []#For predictions
#Perform LASSO
t_start = time.time()
for model_k in range(10):
    #Set training/testing dataset in model #k
    tempL_x = list(x_folds)
    tempDF_x_test = tempL_x.pop(model_k)
    tempA_x_train = np.concatenate(tempL_x)
    tempL_y = list(y_folds)
    tempDF_y_test = tempL_y.pop(model_k)
    tempA_y_train = np.concatenate(tempL_y)
    #Cross-validation with training dataset
    model.fit(tempA_x_train, tempA_y_train)
    #Save parameters (Skip intercepts and alphas here)
    combiBMI_M_bcoefs['Model_'+str(model_k)] = list(model.coef_)#w in the cost function formula
    #Evaluation with testing dataset
    combiBMI_M_scores.append(model.score(tempDF_x_test, tempDF_y_test))
    #Prediction with model #k for testing dataset
    tempL.append(model.predict(tempDF_x_test).flatten())
t_elapsed = time.time() - t_start
print('Elapsed time for 10 models of 10-fold CV LASSO:',
      round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean the predicted values
tempS = pd.Series([value for sublist in tempL for value in sublist],
                  index=tempDF.index, name='log_BaseCombiBMI')
combiBMI_M = pd.concat([tempDF, tempS], axis=1)
#Convert to original scale
combiBMI_M['BaseBMI'] = np.e**combiBMI_M['log_BaseBMI']
combiBMI_M['BaseCombiBMI'] = np.e**combiBMI_M['log_BaseCombiBMI']

display(combiBMI_M)
#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_combiBMI-Male.csv'
combiBMI_M.to_csv(fileDir+fileName, index=True)

In [ ]:
#Both sex model
#Split into 10 datasets
x_folds = np.array_split(combiDF_B_x, 10)
tempDF = combiDF_B.drop(columns=list(set(combiDF_B.columns)-set(['log_BaseBMI'])))#Not Series but DF
y_folds = np.array_split(tempDF, 10)

model = LassoCV(eps=0.01, n_alphas=200, alphas=None, fit_intercept=True,
                normalize=False, precompute='auto', cv=10)
combiBMI_B_bcoefs = pd.DataFrame(index=combiDF_B_x.columns).astype('float64')#For beta-coefficient
combiBMI_B_bcoefs.index.set_names('Variable', inplace=True)
combiBMI_B_scores = []#For the coefficient of determination R^2
tempL = []#For predictions
#Perform LASSO
t_start = time.time()
for model_k in range(10):
    #Set training/testing dataset in model #k
    tempL_x = list(x_folds)
    tempDF_x_test = tempL_x.pop(model_k)
    tempA_x_train = np.concatenate(tempL_x)
    tempL_y = list(y_folds)
    tempDF_y_test = tempL_y.pop(model_k)
    tempA_y_train = np.concatenate(tempL_y)
    #Cross-validation with training dataset
    model.fit(tempA_x_train, tempA_y_train)
    #Save parameters (Skip intercepts and alphas here)
    combiBMI_B_bcoefs['Model_'+str(model_k)] = list(model.coef_)#w in the cost function formula
    #Evaluation with testing dataset
    combiBMI_B_scores.append(model.score(tempDF_x_test, tempDF_y_test))
    #Prediction with model #k for testing dataset
    tempL.append(model.predict(tempDF_x_test).flatten())
t_elapsed = time.time() - t_start
print('Elapsed time for 10 models of 10-fold CV LASSO:',
      round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean the predicted values
tempS = pd.Series([value for sublist in tempL for value in sublist],
                  index=tempDF.index, name='log_BaseCombiBMI')
combiBMI_B = pd.concat([tempDF, tempS], axis=1)
#Convert to original scale
combiBMI_B['BaseBMI'] = np.e**combiBMI_B['log_BaseBMI']
combiBMI_B['BaseCombiBMI'] = np.e**combiBMI_B['log_BaseCombiBMI']

display(combiBMI_B)
#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_combiBMI-BothSex.csv'
combiBMI_B.to_csv(fileDir+fileName, index=True)

### 4-4. Prediction accuracy

In [ ]:
print('Female model')
print('• R2 Score in LASSO [Mean ± SD]:',
      np.mean(combiBMI_F_scores), '±', np.std(combiBMI_F_scores, ddof=1))#Sample standard deviation
print('• R2 Score in LASSO [Mean ± SEM]:',
      np.mean(combiBMI_F_scores), '±', np.std(combiBMI_F_scores, ddof=1)/np.sqrt(10))
print('• Observed vs. LASSO-predicted BMI (log) Pearson\'s r:',
      stats.pearsonr(combiBMI_F['log_BaseBMI'], combiBMI_F['log_BaseCombiBMI']))#output: Pearson's r, p-value
print('• Observed vs. LASSO-predicted BMI Pearson\'s r:',
      stats.pearsonr(combiBMI_F['BaseBMI'], combiBMI_F['BaseCombiBMI']))#output: Pearson's r, p-value
display(combiBMI_F.describe())

print('Male model')
print('• R2 Score in LASSO [Mean ± SD]:',
      np.mean(combiBMI_M_scores), '±', np.std(combiBMI_M_scores, ddof=1))#Sample standard deviation
print('• R2 Score in LASSO [Mean ± SEM]:',
      np.mean(combiBMI_M_scores), '±', np.std(combiBMI_M_scores, ddof=1)/np.sqrt(10))
print('• Observed vs. LASSO-predicted BMI (log) Pearson\'s r:',
      stats.pearsonr(combiBMI_M['log_BaseBMI'], combiBMI_M['log_BaseCombiBMI']))#output: Pearson's r, p-value
print('• Observed vs. LASSO-predicted BMI Pearson\'s r:',
      stats.pearsonr(combiBMI_M['BaseBMI'], combiBMI_M['BaseCombiBMI']))#output: Pearson's r, p-value
display(combiBMI_M.describe())

print('Both sex model')
print('• R2 Score in LASSO [Mean ± SD]:',
      np.mean(combiBMI_B_scores), '±', np.std(combiBMI_B_scores, ddof=1))#Sample standard deviation
print('• R2 Score in LASSO [Mean ± SEM]:',
      np.mean(combiBMI_B_scores), '±', np.std(combiBMI_B_scores, ddof=1)/np.sqrt(10))
print('• Observed vs. LASSO-predicted BMI (log) Pearson\'s r:',
      stats.pearsonr(combiBMI_B['log_BaseBMI'], combiBMI_B['log_BaseCombiBMI']))#output: Pearson's r, p-value
print('• Observed vs. LASSO-predicted BMI Pearson\'s r:',
      stats.pearsonr(combiBMI_B['BaseBMI'], combiBMI_B['BaseCombiBMI']))#output: Pearson's r, p-value
display(combiBMI_B.describe())

In [ ]:
tempDF = pd.DataFrame({'Female':combiBMI_F_scores, 'Male':combiBMI_M_scores, 'Both sex':combiBMI_B_scores})
tempDF = tempDF.melt(var_name='Cohort', value_name='R2score', value_vars=tempDF.columns.tolist())

#Plot
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(3, 1))
sns.barplot(data=tempDF, y='Cohort', x='R2score', hue='Cohort', palette='Set1', dodge=False, edgecolor='black',
            ci=68, capsize=0.4, errwidth=1.5, errcolor='black')#68%CI=Mean±SEM
sns.stripplot(data=tempDF, y='Cohort', x='R2score', hue='Cohort', dodge=False, size=8, edgecolor='black',
              linewidth=1, alpha=0.4, palette={'Female':'gray', 'Male':'gray', 'Both sex':'gray'})
sns.despine()
plt.xlabel('Out-of-sample '+r'$R^2$'+' in 10 LASSO models\n[mean ± SEM]')
plt.ylabel('')
plt.legend('', frameon=False)
plt.show()

In [ ]:
tempDF = pd.concat([combiBMI_F, combiBMI_M, combiBMI_B], axis=0)
tempDF['Model'] = np.repeat(['Female', 'Male', 'Both sex'],
                            [len(combiBMI_F), len(combiBMI_M), len(combiBMI_B)])

#Plot all models of proteomics
sns.set(style='ticks', font='Arial', context='notebook')
sns.lmplot(data=tempDF, x='BaseCombiBMI', y='BaseBMI', hue='Model', palette='Set1',
           scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=2.5, aspect=1)
plt.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))#Y=X as reference
plt.xlabel('Predicted BMI [kg m'+r'$^{-2}$'+']')
plt.ylabel('Measured BMI [kg m'+r'$^{-2}$'+']')
plt.show()

#Plot difference b/w sex
sns.set(style='ticks', font='Arial', context='talk')
sns.lmplot(data=tempDF[tempDF['Model']!='Both sex'], x='BaseCombiBMI', y='BaseBMI',
           hue='Model', palette='Set1', scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=4
          ).set(xlim=(8, 57), xticks=np.arange(10, 51, 10), ylim=(8, 57), yticks=np.arange(10, 51, 10))
plt.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))#Y=X as reference
plt.xlabel('Predicted BMI [kg m'+r'$^{-2}$'+']')
plt.ylabel('Measured BMI [kg m'+r'$^{-2}$'+']')
plt.show()

#Plot only both-sex model
sns.set(style='ticks', font='Arial', context='talk')
sns.lmplot(data=combiBMI_B, x='BaseCombiBMI', y='BaseBMI', line_kws={'color':'m'},
           scatter_kws={'alpha':0.2, 'edgecolor':'black', 'color':'m'}, height=4
          ).set(xlim=(8, 57), xticks=np.arange(10, 51, 10), ylim=(8, 57), yticks=np.arange(10, 51, 10))
plt.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))#Y=X as reference
plt.xlabel('Predicted BMI [kg m'+r'$^{-2}$'+']')
plt.ylabel('Measured BMI [kg m'+r'$^{-2}$'+']')
plt.show()

#Plot difference b/w sex-combined and sex-stratified models
tempS = combiDF_B['Sex']
tempDF = pd.merge(tempDF, tempS, left_index=True, right_index=True, how='left')
tempD = {'Female':'Sex-stratified', 'Male':'Sex-stratified', 'Both sex':'Sex-combined'}
tempDF['Model'] = tempDF['Model'].map(tempD)
tempDF = tempDF.reset_index().set_index(['public_client_id', 'Sex']).pivot(columns='Model')['BaseCombiBMI']
tempDF = tempDF.reset_index()
tempD = {'F':'Female', 'M':'Male'}
sns.set(style='ticks', font='Arial', context='talk')
p = sns.lmplot(data=tempDF, x='Sex-stratified', y='Sex-combined',
               hue='Sex', hue_order=tempD.keys(), palette='Set1', legend=False,
               col='Sex', col_order=tempD.keys(), col_wrap=2,
               scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=4, aspect=1)
for ax, sex in zip(p.axes.ravel(), tempD.keys()):
    #Draw Y=X as reference
    ax.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))
    #Draw Pearson's r
    tempS1 = tempDF.loc[tempDF['Sex']==sex]['Sex-stratified']
    tempS2 = tempDF.loc[tempDF['Sex']==sex]['Sex-combined']
    pearson_r = stats.pearsonr(tempS1, tempS2)[0]
    ax.annotate('Pearson\'s '+r'$r$'+' = '+str(round(pearson_r, 3)), (10, 50), fontsize='small')
    #Change facet label
    ax.set_title(tempD[sex], {'fontsize':'large'})
p.set_axis_labels('Sex-stratified bBMI [kg m'+r'$^{-2}$'+']',
                  'Sex-combined bBMI [kg m'+r'$^{-2}$'+']')
plt.show()

### 4-5. Cleaning beta-coefficient dataframe

In [ ]:
#Female model
tempDF = combiBMI_F_bcoefs.copy(deep=True)
print('Variables:', len(tempDF))

#Summarize
tempL1 = []
tempL2 = []
tempL3 = []
for row_n in tempDF.index.tolist():
    tempL1.append(tempDF.loc[row_n].mean())
    tempL2.append(tempDF.loc[row_n].std(ddof=1))#Sample standard deviation
    tempL3.append((tempDF.loc[row_n]==0.0).astype('int64').sum())
tempDF['MEAN'] = tempL1
tempDF['SD'] = tempL2
tempDF['nZeros'] = tempL3

#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_combiBMI-Female_LASSO-bcoefs.tsv'
tempDF.to_csv(fileDir+fileName, sep='\t', index=True)

#Variables with non-zero beta-coefficient
tempDF = tempDF.loc[tempDF['nZeros']!=10]
print('Variables with non-zero beta-coefficient:', len(tempDF))

#Update
combiBMI_F_bcoefs = tempDF

#Extract robust beta-coefficient: no zeros in all 10 models
tempDF = combiBMI_F_bcoefs.loc[combiBMI_F_bcoefs['nZeros']==0]
tempDF = tempDF.sort_values(by='MEAN', ascending=False)
print('Variables with non-zero beta-coefficient in all 10 models:', len(tempDF))
display(tempDF)

In [ ]:
#Male model
tempDF = combiBMI_M_bcoefs.copy(deep=True)
print('Variables:', len(tempDF))

#Summarize
tempL1 = []
tempL2 = []
tempL3 = []
for row_n in tempDF.index.tolist():
    tempL1.append(tempDF.loc[row_n].mean())
    tempL2.append(tempDF.loc[row_n].std(ddof=1))#Sample standard deviation
    tempL3.append((tempDF.loc[row_n]==0.0).astype('int64').sum())
tempDF['MEAN'] = tempL1
tempDF['SD'] = tempL2
tempDF['nZeros'] = tempL3

#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_combiBMI-Male_LASSO-bcoefs.tsv'
tempDF.to_csv(fileDir+fileName, sep='\t', index=True)

#Variables with non-zero beta-coefficient
tempDF = tempDF.loc[tempDF['nZeros']!=10]
print('Variables with non-zero beta-coefficient:', len(tempDF))

#Update
combiBMI_M_bcoefs = tempDF

#Extract robust beta-coefficient: no zeros in all 10 models
tempDF = combiBMI_M_bcoefs.loc[combiBMI_M_bcoefs['nZeros']==0]
tempDF = tempDF.sort_values(by='MEAN', ascending=False)
print('Variables with non-zero beta-coefficient in all 10 models:', len(tempDF))
display(tempDF)

In [ ]:
#Both sex model
tempDF = combiBMI_B_bcoefs.copy(deep=True)
print('Variables:', len(tempDF))

#Summarize
tempL1 = []
tempL2 = []
tempL3 = []
for row_n in tempDF.index.tolist():
    tempL1.append(tempDF.loc[row_n].mean())
    tempL2.append(tempDF.loc[row_n].std(ddof=1))#Sample standard deviation
    tempL3.append((tempDF.loc[row_n]==0.0).astype('int64').sum())
tempDF['MEAN'] = tempL1
tempDF['SD'] = tempL2
tempDF['nZeros'] = tempL3

#Save
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_combiBMI-BothSex_LASSO-bcoefs.tsv'
tempDF.to_csv(fileDir+fileName, sep='\t', index=True)

#Variables with non-zero beta-coefficient
tempDF = tempDF.loc[tempDF['nZeros']!=10]
print('Variables with non-zero beta-coefficient:', len(tempDF))

#Update
combiBMI_B_bcoefs = tempDF

#Extract robust beta-coefficient: no zeros in all 10 models
tempDF = combiBMI_B_bcoefs.loc[combiBMI_B_bcoefs['nZeros']==0]
tempDF = tempDF.sort_values(by='MEAN', ascending=False)
print('Variables with non-zero beta-coefficient in all 10 models:', len(tempDF))
display(tempDF)

## 5. Comparison b/w models

In [ ]:
tempDF = pd.DataFrame({'Metabolomics':metBMI_B_scores, 'Proteomics':protBMI_B_scores,
                       'Clinical labs':chemBMI_B_scores, 'Combined omics':combiBMI_B_scores})
tempDF = tempDF.melt(var_name='Category', value_name='R2score', value_vars=tempDF.columns.tolist())

#Plot
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(3, 4))
sns.barplot(data=tempDF, x='Category', y='R2score', hue='Category', dodge=False, edgecolor='black',
            palette={'Metabolomics':'b', 'Proteomics':'r', 'Clinical labs':'g', 'Combined omics':'m'},
            ci=68, capsize=0.4, errwidth=1.5, errcolor='black')#68%CI=Mean±SEM
sns.stripplot(data=tempDF, x='Category', y='R2score', hue='Category', dodge=False, size=8, edgecolor='black',
              palette={'Metabolomics':'gray', 'Proteomics':'gray', 'Clinical labs':'gray', 'Combined omics':'gray'},
              linewidth=1, alpha=0.4).set(ylim=(0, 0.85), yticks=np.arange(0, 0.85, 0.2))
sns.despine()
plt.xticks(rotation=70, horizontalalignment='right', verticalalignment='center', rotation_mode='anchor')
plt.ylabel('Out-of-sample '+r'$R^2$')
plt.xlabel('')
plt.legend('', frameon=False)
##Save
fileDir = './ExportFigures/'
fileName = '210530_Biological-BMI-paper_baseline-LASSO_R2score-comparison-BothSex.tif'
plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                  pil_kwargs={'compression':'tiff_lzw'})
plt.show()

In [ ]:
#Plot sex-dependent models as a single figure
tempL = ['Female', 'Male']
sns.set(style='ticks', font='Arial', context='talk')
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(6.5, 4), sharex=True, sharey=True)
for ax_i, ax in enumerate(axes.flat):
    sex = tempL[ax_i]
    if sex=='Female':
        tempDF = pd.DataFrame({'Metabolomics':metBMI_F_scores, 'Proteomics':protBMI_F_scores,
                               'Clinical labs':chemBMI_F_scores, 'Combined omics':combiBMI_F_scores})
    elif sex=='Male':
        tempDF = pd.DataFrame({'Metabolomics':metBMI_M_scores, 'Proteomics':protBMI_M_scores,
                               'Clinical labs':chemBMI_M_scores, 'Combined omics':combiBMI_M_scores})
    tempDF = tempDF.melt(var_name='Category', value_name='R2score', value_vars=tempDF.columns.tolist())
    sns.barplot(data=tempDF, x='Category', y='R2score', hue='Category', dodge=False, edgecolor='black',
                palette={'Metabolomics':'b', 'Proteomics':'r', 'Clinical labs':'g', 'Combined omics':'m'},
                ci=68, capsize=0.4, errwidth=1.5, errcolor='black', ax=ax)#68%CI=Mean±SEM
    sns.stripplot(data=tempDF, x='Category', y='R2score', hue='Category', dodge=False, size=8, edgecolor='black',
                  palette={'Metabolomics':'gray', 'Proteomics':'gray', 'Clinical labs':'gray', 'Combined omics':'gray'},
                  linewidth=1, alpha=0.4, ax=ax)
    sns.despine()
    plt.setp(ax.get_xticklabels(), rotation=70,
             horizontalalignment='right', verticalalignment='center', rotation_mode='anchor')
    if ax_i==0:
        plt.setp(ax, xlabel='', ylabel='Out-of-sample '+r'$R^2$')
    elif ax_i==1:
        plt.setp(ax.get_yticklabels(), visible=False)
        plt.setp(ax, xlabel='', ylabel='')
    ax.set_title(sex, {'fontsize':'large'})
    ax.legend('', frameon=False)
plt.setp(axes, ylim=(0, 0.85), yticks=np.arange(0, 0.85, 0.2))
##Save
fileDir = './ExportFigures/'
fileName = '210530_Biological-BMI-paper_baseline-LASSO_R2score-comparison-FemaleMale.tif'
plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                  pil_kwargs={'compression':'tiff_lzw'})
plt.show()

In [ ]:
tempDF = combiBMI_B.drop(columns=['log_BaseCombiBMI', 'BaseCombiBMI'])
tempL = [metBMI_B, protBMI_B, chemBMI_B, combiBMI_B]
for df in tempL:
    tempDF = pd.merge(tempDF, df.drop(columns=['log_BaseBMI', 'BaseBMI']),
                      left_index=True, right_index=True, how='inner')
tempDF = tempDF.loc[:, ~tempDF.columns.str.contains(pat='log_')]
tempDF = tempDF.reset_index().melt(var_name='Model', value_name='bBMI',
                                   id_vars=['public_client_id', 'BaseBMI'])

#Plot
tempD = {'BaseMetBMI':'b', 'BaseProtBMI':'r', 'BaseChemBMI':'g', 'BaseCombiBMI':'m'}
sns.set(style='ticks', font='Arial', context='talk')
p = sns.lmplot(data=tempDF, x='bBMI', y='BaseBMI', hue='Model', hue_order=tempD.keys(), palette=tempD,
               col='Model', col_wrap=2,  col_order=tempD.keys(), legend=False, 
               scatter_kws={'alpha':0.2, 'edgecolor':'black'}, height=4, aspect=1)
p.set(xlim=(8, 57), xticks=np.arange(10, 51, 10), ylim=(8, 57), yticks=np.arange(10, 51, 10))
tempD = {'BaseMetBMI':'Metabolomics', 'BaseProtBMI':'Proteomics',
         'BaseChemBMI':'Clinical labs', 'BaseCombiBMI':'Combined omics'}
for ax, bmi in zip(p.axes.ravel(), tempD.keys()):
    #Draw Y=X as reference
    ax.plot([10, 55], [10, 55], color='black', linestyle=(0, (1, 2)))
    #Draw Pearson's r
    tempS1 = tempDF.loc[tempDF['Model']==bmi]['BaseBMI']
    tempS2 = tempDF.loc[tempDF['Model']==bmi]['bBMI']
    pearson_r = stats.pearsonr(tempS1, tempS2)[0]
    ax.annotate('Pearson\'s '+r'$r$'+' = '+str(round(pearson_r, 3)), (10, 50), fontsize='small')
    #Change facet label
    ax.set_title(tempD[bmi], {'fontsize':'large'})
#Reset and generate common axis title
p.set_axis_labels('', '')#Empty box -> va=center for xlabel, ha=center for ylabel
p.fig.text(x=0.5, y=0, s='Predicted BMI [kg m'+r'$^{-2}$'+']', fontsize='medium',
           verticalalignment='center', horizontalalignment='center')
p.fig.text(x=0, y=0.5, s='Measured BMI [kg m'+r'$^{-2}$'+']', fontsize='medium',
           verticalalignment='center', horizontalalignment='center', rotation='vertical')
##Save
fileDir = './ExportFigures/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_scatterplot-comparison-BothSex.tif'
plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                  pil_kwargs={'compression':'tiff_lzw'})
plt.show()

## 6. Performance transition vs. critical variables

> If eliminating critical variables during LASSO, performance should be dropped.  
> —> Repeat LASSO while dropping the top variable.  
>
> Due to time consuming calculation, skip female/male cohort version.

### 6-1. Metabolomics

In [ ]:
print('Start:', time.ctime(time.time()))
dropL = []
scoresDF = pd.DataFrame(columns=['Model_'+str(k) for k in range(10)]).astype('float64')
scoresDF.index.set_names('Iteration', inplace=True)
for iter_n in range(101):#range(len(metDF_B_x.columns)):
    print('Iteration', iter_n)
    t_start = time.time()
    #Drop the top variable
    tempDF = metDF_B_x.drop(columns=dropL)
    tempL = tempDF.columns.tolist()
    #Split into 10 datasets
    x_folds = np.array_split(tempDF, 10)
    tempDF = metDF_B.drop(columns=list(set(metDF_B.columns)-set(['log_BaseBMI'])))#Not Series but DF
    y_folds = np.array_split(tempDF, 10)
    #Perform LASSO
    model = LassoCV(eps=0.05, n_alphas=200, alphas=None, fit_intercept=True,
                    normalize=False, precompute='auto', cv=10)
    tempDF = pd.DataFrame(index=tempL).astype('float64')#For beta-coefficient
    tempDF.index.set_names('Variable', inplace=True)
    tempL = []#For the coefficient of determination R^2
    for model_k in range(10):
        #Set training/testing dataset in model #k
        tempL_x = list(x_folds)
        tempDF_x_test = tempL_x.pop(model_k)
        tempA_x_train = np.concatenate(tempL_x)
        tempL_y = list(y_folds)
        tempDF_y_test = tempL_y.pop(model_k)
        tempA_y_train = np.concatenate(tempL_y)
        #Cross-validation with training dataset
        model.fit(tempA_x_train, tempA_y_train)
        #Save parameters (Skip intercepts and alphas here)
        tempDF[model_k] = list(model.coef_)#w in the cost function formula
        #Evaluation with testing dataset
        tempL.append(model.score(tempDF_x_test, tempDF_y_test))
    #Prediction accuracy
    print(' • R2 Score in LASSO [Mean ± SEM]:', np.mean(tempL), '±', np.std(tempL, ddof=1)/np.sqrt(10))
    if np.mean(tempL) < 0:
        print('-> Finish')
        break
    scoresDF.loc['Iteration_'+str(iter_n)] = tempL
    #Cleaning beta-coefficients dataframe
    ##Identify all variables with non-zero beta-coefficient in all 10 models
    for row_n in tempDF.index.tolist():
        if (tempDF.loc[row_n]==0.0).sum() != 0:
            tempDF.drop(index=[row_n], inplace=True)
    ##Calculate mean beta-coefficient for the robust variables
    tempS = tempDF.mean(axis=1)
    tempS = np.abs(tempS)
    tempS = tempS.sort_values(ascending=False)
    print(' • Top variable:', tempS.index[0])
    dropL.append(tempS.index[0])
    t_elapsed = time.time() - t_start
    print(' • Elapsed time:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')
print('Finish:', time.ctime(time.time()))

#Save just in case for connection error
scoresDF['TopVariable'] = dropL
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_metBMI-BothSex_R2transition.tsv'
scoresDF.to_csv(fileDir+fileName, sep='\t', index=True)

In [ ]:
#Load scoresDF just in case for connection error
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_metBMI-BothSex_R2transition.tsv'
tempDF = pd.read_csv(fileDir+fileName, sep='\t').set_index('Iteration')
tempS = tempDF['TopVariable']

#Plot performance transition
tempDF = tempDF.drop(columns=['TopVariable'])
tempDF['IterNum'] = list(range(len(tempDF)))
tempDF = tempDF.reset_index().melt(var_name='Model', value_name='R2', id_vars=['Iteration', 'IterNum'])
##Prepare original robust variables to judge performance in LASSO
tempL = metBMI_B_bcoefs.loc[metBMI_B_bcoefs['nZeros']==0].index.tolist()

sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(6, 4))
sns.lineplot(data=tempDF, x='IterNum', y='R2', estimator='mean', ci=68, **{'color':'black'}#68%CI=Mean±SEM
            ).set(xlim=(0, 100), ylim=(0, 0.8), yticks=np.arange(0, 0.81, 0.2))
sns.despine()
plt.xlabel('Dropping iteration number')
plt.ylabel('Out-of-sample '+r'$R^2$')
for iter_i in range(len(tempS)):
    if tempS.iloc[iter_i] in tempL:
        plt.axvspan(xmin=iter_i, xmax=iter_i+1, facecolor='b', alpha=0.4)
plt.margins(0, 0, tight=True)
##Save
fileDir = './ExportFigures/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_metBMI-BothSex_R2transition.tif'
plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                  pil_kwargs={'compression':'tiff_lzw'})
plt.show()

### 6-2. Clinical labs

In [ ]:
print('Start:', time.ctime(time.time()))
dropL = []
scoresDF = pd.DataFrame(columns=['Model_'+str(k) for k in range(10)]).astype('float64')
scoresDF.index.set_names('Iteration', inplace=True)
for iter_n in range(101):#range(len(chemDF_B_x.columns)):
    print('Iteration', iter_n)
    t_start = time.time()
    #Drop the top variable
    tempDF = chemDF_B_x.drop(columns=dropL)
    tempL = tempDF.columns.tolist()
    #Split into 10 datasets
    x_folds = np.array_split(tempDF, 10)
    tempDF = chemDF_B.drop(columns=list(set(chemDF_B.columns)-set(['log_BaseBMI'])))#Not Series but DF
    y_folds = np.array_split(tempDF, 10)
    #Perform LASSO
    model = LassoCV(eps=0.05, n_alphas=200, alphas=None, fit_intercept=True,
                    normalize=False, precompute='auto', cv=10)
    tempDF = pd.DataFrame(index=tempL).astype('float64')#For beta-coefficient
    tempDF.index.set_names('Variable', inplace=True)
    tempL = []#For the coefficient of determination R^2
    for model_k in range(10):
        #Set training/testing dataset in model #k
        tempL_x = list(x_folds)
        tempDF_x_test = tempL_x.pop(model_k)
        tempA_x_train = np.concatenate(tempL_x)
        tempL_y = list(y_folds)
        tempDF_y_test = tempL_y.pop(model_k)
        tempA_y_train = np.concatenate(tempL_y)
        #Cross-validation with training dataset
        model.fit(tempA_x_train, tempA_y_train)
        #Save parameters (Skip intercepts and alphas here)
        tempDF[model_k] = list(model.coef_)#w in the cost function formula
        #Evaluation with testing dataset
        tempL.append(model.score(tempDF_x_test, tempDF_y_test))
    #Prediction accuracy
    print(' • R2 Score in LASSO [Mean ± SEM]:', np.mean(tempL), '±', np.std(tempL, ddof=1)/np.sqrt(10))
    if np.mean(tempL) < 0:
        print('-> Finish')
        break
    scoresDF.loc['Iteration_'+str(iter_n)] = tempL
    #Cleaning beta-coefficients dataframe
    ##Identify all variables with non-zero beta-coefficient in all 10 models
    for row_n in tempDF.index.tolist():
        if (tempDF.loc[row_n]==0.0).sum() != 0:
            tempDF.drop(index=[row_n], inplace=True)
    ##Calculate mean beta-coefficient for the robust variables
    tempS = tempDF.mean(axis=1)
    tempS = np.abs(tempS)
    tempS = tempS.sort_values(ascending=False)
    print(' • Top variable:', tempS.index[0])
    dropL.append(tempS.index[0])
    t_elapsed = time.time() - t_start
    print(' • Elapsed time:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')
print('Finish:', time.ctime(time.time()))

#Save just in case for connection error
scoresDF['TopVariable'] = dropL
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_chemBMI-BothSex_R2transition.tsv'
scoresDF.to_csv(fileDir+fileName, sep='\t', index=True)

In [ ]:
#Load scoresDF just in case for connection error
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_chemBMI-BothSex_R2transition.tsv'
tempDF = pd.read_csv(fileDir+fileName, sep='\t').set_index('Iteration')
tempS = tempDF['TopVariable']

#Plot performance transition
tempDF = tempDF.drop(columns=['TopVariable'])
tempDF['IterNum'] = list(range(len(tempDF)))
tempDF = tempDF.reset_index().melt(var_name='Model', value_name='R2', id_vars=['Iteration', 'IterNum'])
##Prepare original robust variables to judge performance in LASSO
tempL = chemBMI_B_bcoefs.loc[chemBMI_B_bcoefs['nZeros']==0].index.tolist()

sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(6, 4))
sns.lineplot(data=tempDF, x='IterNum', y='R2', estimator='mean', ci=68, **{'color':'black'}#68%CI=Mean±SEM
            ).set(xlim=(0, 100), ylim=(0, 0.8), yticks=np.arange(0, 0.81, 0.2))
sns.despine()
plt.xlabel('Dropping iteration number')
plt.ylabel('Out-of-sample '+r'$R^2$')
for iter_i in range(len(tempS)):
    if tempS.iloc[iter_i] in tempL:
        plt.axvspan(xmin=iter_i, xmax=iter_i+1, facecolor='g', alpha=0.4)
plt.margins(0, 0, tight=True)
##Save
fileDir = './ExportFigures/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_chemBMI-BothSex_R2transition.tif'
plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                  pil_kwargs={'compression':'tiff_lzw'})
plt.show()

### 6-3. Proteomics

In [ ]:
print('Start:', time.ctime(time.time()))
dropL = []
scoresDF = pd.DataFrame(columns=['Model_'+str(k) for k in range(10)]).astype('float64')
scoresDF.index.set_names('Iteration', inplace=True)
for iter_n in range(101):#range(len(protDF_B_x.columns)):
    print('Iteration', iter_n)
    t_start = time.time()
    #Drop the top variable
    tempDF = protDF_B_x.drop(columns=dropL)
    tempL = tempDF.columns.tolist()
    #Split into 10 datasets
    x_folds = np.array_split(tempDF, 10)
    tempDF = protDF_B.drop(columns=list(set(protDF_B.columns)-set(['log_BaseBMI'])))#Not Series but DF
    y_folds = np.array_split(tempDF, 10)
    #Perform LASSO
    model = LassoCV(eps=0.05, n_alphas=200, alphas=None, fit_intercept=True,
                    normalize=False, precompute='auto', cv=10)
    tempDF = pd.DataFrame(index=tempL).astype('float64')#For beta-coefficient
    tempDF.index.set_names('Variable', inplace=True)
    tempL = []#For the coefficient of determination R^2
    for model_k in range(10):
        #Set training/testing dataset in model #k
        tempL_x = list(x_folds)
        tempDF_x_test = tempL_x.pop(model_k)
        tempA_x_train = np.concatenate(tempL_x)
        tempL_y = list(y_folds)
        tempDF_y_test = tempL_y.pop(model_k)
        tempA_y_train = np.concatenate(tempL_y)
        #Cross-validation with training dataset
        model.fit(tempA_x_train, tempA_y_train)
        #Save parameters (Skip intercepts and alphas here)
        tempDF[model_k] = list(model.coef_)#w in the cost function formula
        #Evaluation with testing dataset
        tempL.append(model.score(tempDF_x_test, tempDF_y_test))
    #Prediction accuracy
    print(' • R2 Score in LASSO [Mean ± SEM]:', np.mean(tempL), '±', np.std(tempL, ddof=1)/np.sqrt(10))
    if np.mean(tempL) < 0:
        print('-> Finish')
        break
    scoresDF.loc['Iteration_'+str(iter_n)] = tempL
    #Cleaning beta-coefficients dataframe
    ##Identify all variables with non-zero beta-coefficient in all 10 models
    for row_n in tempDF.index.tolist():
        if (tempDF.loc[row_n]==0.0).sum() != 0:
            tempDF.drop(index=[row_n], inplace=True)
    ##Calculate mean beta-coefficient for the robust variables
    tempS = tempDF.mean(axis=1)
    tempS = np.abs(tempS)
    tempS = tempS.sort_values(ascending=False)
    print(' • Top variable:', tempS.index[0])
    dropL.append(tempS.index[0])
    t_elapsed = time.time() - t_start
    print(' • Elapsed time:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')
print('Finish:', time.ctime(time.time()))

#Save just in case for connection error
scoresDF['TopVariable'] = dropL
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_protBMI-BothSex_R2transition.tsv'
scoresDF.to_csv(fileDir+fileName, sep='\t', index=True)

In [ ]:
#Load scoresDF just in case for connection error
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_protBMI-BothSex_R2transition.tsv'
tempDF = pd.read_csv(fileDir+fileName, sep='\t').set_index('Iteration')
tempS = tempDF['TopVariable']

#Plot performance transition
tempDF = tempDF.drop(columns=['TopVariable'])
tempDF['IterNum'] = list(range(len(tempDF)))
tempDF = tempDF.reset_index().melt(var_name='Model', value_name='R2', id_vars=['Iteration', 'IterNum'])
##Prepare original robust variables to judge performance in LASSO
tempL = protBMI_B_bcoefs.loc[protBMI_B_bcoefs['nZeros']==0].index.tolist()

sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(6, 4))
sns.lineplot(data=tempDF, x='IterNum', y='R2', estimator='mean', ci=68, **{'color':'black'}#68%CI=Mean±SEM
            ).set(xlim=(0, 100), ylim=(0, 0.8), yticks=np.arange(0, 0.81, 0.2))
sns.despine()
plt.xlabel('Dropping iteration number')
plt.ylabel('Out-of-sample '+r'$R^2$')
for iter_i in range(len(tempS)):
    if tempS.iloc[iter_i] in tempL:
        plt.axvspan(xmin=iter_i, xmax=iter_i+1, facecolor='r', alpha=0.4)
plt.margins(0, 0, tight=True)
##Save
fileDir = './ExportFigures/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_protBMI-BothSex_R2transition.tif'
plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                  pil_kwargs={'compression':'tiff_lzw'})
plt.show()

### 6-4. Combined omics

In [ ]:
print('Start:', time.ctime(time.time()))
dropL = []
scoresDF = pd.DataFrame(columns=['Model_'+str(k) for k in range(10)]).astype('float64')
scoresDF.index.set_names('Iteration', inplace=True)
for iter_n in range(101):#range(len(combiDF_B_x.columns)):
    print('Iteration', iter_n)
    t_start = time.time()
    #Drop the top variable
    tempDF = combiDF_B_x.drop(columns=dropL)
    tempL = tempDF.columns.tolist()
    #Split into 10 datasets
    x_folds = np.array_split(tempDF, 10)
    tempDF = combiDF_B.drop(columns=list(set(combiDF_B.columns)-set(['log_BaseBMI'])))#Not Series but DF
    y_folds = np.array_split(tempDF, 10)
    #Perform LASSO
    model = LassoCV(eps=0.01, n_alphas=200, alphas=None, fit_intercept=True,
                    normalize=False, precompute='auto', cv=10)
    tempDF = pd.DataFrame(index=tempL).astype('float64')#For beta-coefficient
    tempDF.index.set_names('Variable', inplace=True)
    tempL = []#For the coefficient of determination R^2
    for model_k in range(10):
        #Set training/testing dataset in model #k
        tempL_x = list(x_folds)
        tempDF_x_test = tempL_x.pop(model_k)
        tempA_x_train = np.concatenate(tempL_x)
        tempL_y = list(y_folds)
        tempDF_y_test = tempL_y.pop(model_k)
        tempA_y_train = np.concatenate(tempL_y)
        #Cross-validation with training dataset
        model.fit(tempA_x_train, tempA_y_train)
        #Save parameters (Skip intercepts and alphas here)
        tempDF[model_k] = list(model.coef_)#w in the cost function formula
        #Evaluation with testing dataset
        tempL.append(model.score(tempDF_x_test, tempDF_y_test))
    #Prediction accuracy
    print(' • R2 Score in LASSO [Mean ± SEM]:', np.mean(tempL), '±', np.std(tempL, ddof=1)/np.sqrt(10))
    if np.mean(tempL) < 0:
        print('-> Finish')
        break
    scoresDF.loc['Iteration_'+str(iter_n)] = tempL
    #Cleaning beta-coefficients dataframe
    ##Identify all variables with non-zero beta-coefficient in all 10 models
    for row_n in tempDF.index.tolist():
        if (tempDF.loc[row_n]==0.0).sum() != 0:
            tempDF.drop(index=[row_n], inplace=True)
    ##Calculate mean beta-coefficient for the robust variables
    tempS = tempDF.mean(axis=1)
    tempS = np.abs(tempS)
    tempS = tempS.sort_values(ascending=False)
    print(' • Top variable:', tempS.index[0])
    dropL.append(tempS.index[0])
    t_elapsed = time.time() - t_start
    print(' • Elapsed time:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')
print('Finish:', time.ctime(time.time()))

#Save just in case for connection error
scoresDF['TopVariable'] = dropL
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_combiBMI-BothSex_R2transition.tsv'
scoresDF.to_csv(fileDir+fileName, sep='\t', index=True)

In [ ]:
#Load scoresDF just in case for connection error
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_combiBMI-BothSex_R2transition.tsv'
tempDF = pd.read_csv(fileDir+fileName, sep='\t').set_index('Iteration')
tempS = tempDF['TopVariable']

#Plot performance transition
tempDF = tempDF.drop(columns=['TopVariable'])
tempDF['IterNum'] = list(range(len(tempDF)))
tempDF = tempDF.reset_index().melt(var_name='Model', value_name='R2', id_vars=['Iteration', 'IterNum'])
##Prepare original robust variables to judge performance in LASSO
tempL = combiBMI_B_bcoefs.loc[combiBMI_B_bcoefs['nZeros']==0].index.tolist()

sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(6, 4))
sns.lineplot(data=tempDF, x='IterNum', y='R2', estimator='mean', ci=68, **{'color':'black'}#68%CI=Mean±SEM
            ).set(xlim=(0, 100), ylim=(0, 0.8), yticks=np.arange(0, 0.81, 0.2))
sns.despine()
plt.xlabel('Dropping iteration number')
plt.ylabel('Out-of-sample '+r'$R^2$')
for iter_i in range(len(tempS)):
    if tempS.iloc[iter_i] in tempL:
        plt.axvspan(xmin=iter_i, xmax=iter_i+1, facecolor='m', alpha=0.4)
plt.margins(0, 0, tight=True)
##Save
fileDir = './ExportFigures/'
fileName = '210105_Biological-BMI-paper_baseline-LASSO_combiBMI-BothSex_R2transition.tif'
plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                  pil_kwargs={'compression':'tiff_lzw'})
plt.show()

## — End of this notebook —